In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import warnings
import lightgbm as lgb
import json
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
import networkx as nx


from IPython.display import display, Markdown
from IPython.display import display
from datetime import timedelta
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from ta.trend import MACD, SMAIndicator
from ta.momentum import RSIIndicator, ROCIndicator
from ta.volatility import BollingerBands
from typing import Optional, List, Tuple, Dict, Any


# Ignore all warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')



# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 2.5,
})

# Configuration
FORECAST_HORIZONS = [1, 7, 14, 30]
TEST_SIZE_DAYS = 30  # Last 30 days for testing
RANDOM_STATE = 42
LSTM_LOOKBACK = 14
MODEL_DIR = '../models'

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

print("✓ Imports complete")

✓ Imports complete


<br> <br> <br>

### 1. Creating and Imporing Datasets

In [9]:
# Edges
df_edges_plant = pd.read_csv("../data/SupplyGraph/Edges/Edges (Plant).csv")
df_edges_product_group = pd.read_csv("../data/SupplyGraph/Edges/Edges (Product Group).csv")
df_edges_product_subgroup = pd.read_csv("../data/SupplyGraph/Edges/Edges (Product Sub-Group).csv")
df_edges_storage_location = pd.read_csv("../data/SupplyGraph/Edges/Edges (Storage Location).csv")

# Nodes
df_nodes_productgroup_and_subgroup = pd.read_csv("../data/SupplyGraph/Nodes/Node Types (Product Group and Subgroup).csv")
df_nodes_plant_and_storage = pd.read_csv("../data/SupplyGraph/Nodes/Nodes Type (Plant & Storage).csv")
df_nodes = pd.read_csv("../data/SupplyGraph/Nodes/Nodes.csv")
# df_nodes_index = pd.read_csv("../data/SupplyGraph/Nodes/NodesIndex.csv")

# Temporal
df_temporal_delivery_to_distributor = pd.read_csv("../data/SupplyGraph/Temporal Data/Unit/Delivery To distributor.csv")
df_temporal_factory_issue = pd.read_csv("../data/SupplyGraph/Temporal Data/Unit/Factory Issue.csv")
df_temporal_production = pd.read_csv("../data/SupplyGraph/Temporal Data/Unit/Production.csv")
df_temporal_sales_order = pd.read_csv("../data/SupplyGraph/Temporal Data/Unit/Sales Order.csv")

# Pivot the datasets
df_temporal_sales_order['Date'] = pd.to_datetime(df_temporal_sales_order['Date'])
df_temporal_sales_order_pivot = df_temporal_sales_order.melt(id_vars='Date', var_name='Product', value_name='Sales').dropna()
df_temporal_production['Date'] = pd.to_datetime(df_temporal_production['Date'])
df_temporal_production_pivot = df_temporal_production.melt(id_vars='Date', var_name='Product', value_name='Production Quantity').dropna()
df_temporal_factory_issue['Date'] = pd.to_datetime(df_temporal_factory_issue['Date'])
df_temporal_factory_issue_pivot = df_temporal_factory_issue.melt(id_vars='Date', var_name='Product', value_name='Factory Issue').dropna()
df_temporal_delivery_to_distributor['Date'] = pd.to_datetime(df_temporal_delivery_to_distributor['Date'])
df_temporal_delivery_to_distributor_pivot = df_temporal_delivery_to_distributor.melt(id_vars='Date', var_name='Product', value_name='Distributor').dropna()

# Trading Economics Dataset
# daily
df_trading_economics_currency = pd.read_csv("../data/tradingeconomics/Overview/currency.csv", encoding="utf-16")
df_trading_economics_currency["Date"] = pd.to_datetime(df_trading_economics_currency["Date"], dayfirst=True, errors='coerce')

# daily
df_trading_economics_stock_market = pd.read_csv("../data/tradingeconomics/Overview/stock_market.csv", encoding="utf-16")
df_trading_economics_stock_market["Date"] = pd.to_datetime(df_trading_economics_stock_market["Date"], dayfirst=True, errors='coerce')

# daily
df_trading_economics_interest_rate = pd.read_csv("../data/tradingeconomics/Money/Interest_Rate.csv")
df_trading_economics_interest_rate["DateTime"] = pd.to_datetime(df_trading_economics_interest_rate["DateTime"], errors='coerce')
df_trading_economics_interest_rate["LastUpdate"] = pd.to_datetime(df_trading_economics_interest_rate["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_inflation_rate = pd.read_csv("../data/tradingeconomics/Overview/inflation_rate.csv")
df_trading_economics_inflation_rate["DateTime"] = pd.to_datetime(df_trading_economics_inflation_rate["DateTime"], errors='coerce')
df_trading_economics_inflation_rate["LastUpdate"] = pd.to_datetime(df_trading_economics_inflation_rate["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_balance_of_trade = pd.read_csv("../data/tradingeconomics/Overview/balance_of_trade.csv")
df_trading_economics_balance_of_trade["DateTime"] = pd.to_datetime(df_trading_economics_balance_of_trade["DateTime"], errors='coerce')
df_trading_economics_balance_of_trade["LastUpdate"] = pd.to_datetime(df_trading_economics_balance_of_trade["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_export = pd.read_csv("../data/tradingeconomics/Trade/Exports.csv")
df_trading_economics_export["DateTime"] = pd.to_datetime(df_trading_economics_export["DateTime"], errors='coerce')
df_trading_economics_export["LastUpdate"] = pd.to_datetime(df_trading_economics_export["LastUpdate"], errors='coerce')

# monthly
df_trading_economics_import = pd.read_csv("../data/tradingeconomics/Trade/Imports.csv")
df_trading_economics_import["DateTime"] = pd.to_datetime(df_trading_economics_import["DateTime"], errors='coerce')
df_trading_economics_import["LastUpdate"] = pd.to_datetime(df_trading_economics_import["LastUpdate"], errors='coerce')

In [10]:
df_trading_economics_balance_of_trade

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
0,Bangladesh,Balance of Trade,1976-03-31,-1.20,Monthly,BangladeshBT,2014-08-26 18:11:00
1,Bangladesh,Balance of Trade,1976-04-30,-0.60,Monthly,BangladeshBT,2014-08-26 18:11:00
2,Bangladesh,Balance of Trade,1976-05-31,-0.60,Monthly,BangladeshBT,2014-11-16 21:27:00
3,Bangladesh,Balance of Trade,1976-06-30,-1.40,Monthly,BangladeshBT,2014-08-26 18:11:00
4,Bangladesh,Balance of Trade,1976-07-31,-0.70,Monthly,BangladeshBT,2014-08-26 18:11:00
...,...,...,...,...,...,...,...
589,Bangladesh,Balance of Trade,2025-06-30,-181.83,Monthly,BangladeshBT,2025-08-26 05:05:00
590,Bangladesh,Balance of Trade,2025-07-31,-211.58,Monthly,BangladeshBT,2025-09-25 02:42:00
591,Bangladesh,Balance of Trade,2025-08-31,-118.86,Monthly,BangladeshBT,2025-10-29 06:01:00
592,Bangladesh,Balance of Trade,2025-09-30,-203.34,Monthly,BangladeshBT,2026-01-05 06:21:00


In [11]:
df_trading_economics_import[df_trading_economics_import['DateTime'].dt.year == 2023]

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
560,Bangladesh,Imports,2023-01-31,544.95,Monthly,BangladeshIm,2025-03-06 03:07:00
561,Bangladesh,Imports,2023-02-28,447.06,Monthly,BangladeshIm,2025-03-06 03:08:00
562,Bangladesh,Imports,2023-03-31,542.14,Monthly,BangladeshIm,2025-03-06 03:08:00
563,Bangladesh,Imports,2023-04-30,483.87,Monthly,BangladeshIm,2025-03-06 03:09:00
564,Bangladesh,Imports,2023-05-31,587.59,Monthly,BangladeshIm,2025-03-06 03:09:00
565,Bangladesh,Imports,2023-06-30,478.30,Monthly,BangladeshIm,2025-03-06 03:09:00
566,Bangladesh,Imports,2023-07-31,650.47,Monthly,BangladeshIm,2025-03-06 03:09:00
567,Bangladesh,Imports,2023-08-31,590.46,Monthly,BangladeshIm,2025-03-06 03:10:00
568,Bangladesh,Imports,2023-09-30,495.88,Monthly,BangladeshIm,2025-03-06 03:10:00
569,Bangladesh,Imports,2023-10-31,593.47,Monthly,BangladeshIm,2025-03-06 03:10:00


In [12]:
df_trading_economics_export[df_trading_economics_export['DateTime'].dt.year == 2023]

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
610,Bangladesh,Exports,2023-01-31,375.81,Monthly,BangladeshEx,2024-07-17 09:08:00
611,Bangladesh,Exports,2023-02-28,337.08,Monthly,BangladeshEx,2024-07-17 09:08:00
612,Bangladesh,Exports,2023-03-31,357.36,Monthly,BangladeshEx,2024-07-17 09:08:00
613,Bangladesh,Exports,2023-04-30,296.85,Monthly,BangladeshEx,2024-07-17 09:08:00
614,Bangladesh,Exports,2023-05-31,341.64,Monthly,BangladeshEx,2024-07-17 09:08:00
615,Bangladesh,Exports,2023-06-30,322.83,Monthly,BangladeshEx,2024-07-17 09:08:00
616,Bangladesh,Exports,2023-07-31,392.71,Monthly,BangladeshEx,2025-03-06 02:55:00
617,Bangladesh,Exports,2023-08-31,386.11,Monthly,BangladeshEx,2025-03-06 02:55:00
618,Bangladesh,Exports,2023-09-30,313.47,Monthly,BangladeshEx,2025-03-06 02:56:00
619,Bangladesh,Exports,2023-10-31,363.82,Monthly,BangladeshEx,2025-03-06 02:56:00


In [13]:
df_trading_economics_balance_of_trade[df_trading_economics_balance_of_trade['DateTime'].dt.year == 2023]

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
560,Bangladesh,Balance of Trade,2023-01-31,-169.1,Monthly,BangladeshBT,2024-07-17 09:22:00
561,Bangladesh,Balance of Trade,2023-02-28,-110.0,Monthly,BangladeshBT,2024-07-17 09:22:00
562,Bangladesh,Balance of Trade,2023-03-31,-184.8,Monthly,BangladeshBT,2024-07-17 09:22:00
563,Bangladesh,Balance of Trade,2023-04-30,-187.0,Monthly,BangladeshBT,2024-07-17 09:22:00
564,Bangladesh,Balance of Trade,2023-05-31,-245.9,Monthly,BangladeshBT,2024-07-17 09:22:00
565,Bangladesh,Balance of Trade,2023-06-30,-155.5,Monthly,BangladeshBT,2024-07-17 09:22:00
566,Bangladesh,Balance of Trade,2023-07-31,-257.8,Monthly,BangladeshBT,2025-03-05 09:27:00
567,Bangladesh,Balance of Trade,2023-08-31,-204.4,Monthly,BangladeshBT,2025-03-05 09:28:00
568,Bangladesh,Balance of Trade,2023-09-30,-182.4,Monthly,BangladeshBT,2025-03-05 09:28:00
569,Bangladesh,Balance of Trade,2023-10-31,-229.6,Monthly,BangladeshBT,2025-03-05 09:28:00


<br> <br> <br>


#### 1.1 Create a Holiday Dataset



In [14]:
# @misc{BBHolidays2023,
#   title        = {BB Holidays 2023},
#   author       = {{Bangladesh Bank}},
#   year         = {2023},
#   howpublished = {\url{https://www.scribd.com/document/626649862/BB-Holidays-2023}},
#   note         = {Accessed via Scribd}
# }


holidays_dict = [
    {
        'Date': '2023-02-21',
        'holiday_type': "National Holiday",
        'holiday_name': "Language Mother Day"
    },
    {
        'Date': "2023-03-8",
        'holiday_type': "National Holiday",
        'holiday_name': "Shab e-barat"
    },   
    {
        'Date': "2023-03-17",
        'holiday_type': "National Holiday",
        'holiday_name': "Sheikh Mujibur Rahman's birthday",
    },
    {
        'Date': "2023-03-26",
        'holiday_type': "National Holiday",
        'holiday_name': "Independence Day",
    },
    {
        'Date': "2023-04-14",
        'holiday_type': "National Holiday",
        'holiday_name': "Bengali New Year" 
    },
    {
        'Date': "2023-04-19",
        'holiday_type': "National Holiday",
        'holiday_name': "Shab-e-Qadr"
    },
    {
        'Date': "2023-04-21",
        'holiday_type': "National Holiday",
        'holiday_name': "Jumatul Bidah"
    },
    {
        'Date': "2023-04-22",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid al-Fitr"
    },
    {
        'Date': "2023-04-23",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid al-Fitr"
    },
    {
        'Date': "2023-05-01",
        'holiday_type': "National Holiday",
        'holiday_name': "May Day"
    },
    {
        'Date': "2023-05-04",
        'holiday_type': "National Holiday",
        'holiday_name': "Buddha Purnima"
    },
    {
        'Date': "2023-06-28",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-06-29",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-06-30",
        'holiday_type': "National Holiday",
        'holiday_name': "Eid-ul-Azha"
    },
    {
        'Date': "2023-07-01",
        'holiday_type': "Not a National Holiday",
        'holiday_name': "Bank Holiday"
    },
    {
        'Date': "2023-07-29",
        'holiday_type': "National Holiday",
        'holiday_name': "Ashura"
    },
    {
        'Date': "2023-08-15",
        'holiday_type': "National Holiday",
        'holiday_name': "National Mourning Day"
    },       
]


# Create dataframe from holidays_dict
df_holidays = pd.DataFrame(holidays_dict)
df_holidays['Date'] = pd.to_datetime(df_holidays['Date']).dt.normalize()

# Optional: rename for clarity
df_holidays = df_holidays.rename(columns={
    'holiday_name': 'holiday_name',
    'holiday_type': 'holiday_type'
})

# Display the newly created holidays dataframe
display(df_holidays)


,Date,holiday_type,holiday_name
0,2023-02-21,National Holiday,Language Mother Day
1,2023-03-08,National Holiday,Shab e-barat
2,2023-03-17,National Holiday,Sheikh Mujibur Rahman's birthday
3,2023-03-26,National Holiday,Independence Day
4,2023-04-14,National Holiday,Bengali New Year
5,2023-04-19,National Holiday,Shab-e-Qadr
6,2023-04-21,National Holiday,Jumatul Bidah
7,2023-04-22,National Holiday,Eid al-Fitr
8,2023-04-23,National Holiday,Eid al-Fitr
9,2023-05-01,National Holiday,May Day


<br> <br> <br>

#### 1.2 Create a Weather Dataset

In [15]:
# @dataset{zubair2024high,
#   author       = {Zubair, Md and Ishtiaque Mahee, Md Nafiz and Reza, Khondaker Masfiq and Salim, Md Shahidul and Ahmed, Nasim},
#   title        = {High Volume Real-World Weather Data},
#   year         = {2024},
#   publisher    = {Mendeley Data},
#   version      = {V1},
#   doi          = {10.17632/tbrhznpwg9.1},
#   url          = {https://data.mendeley.com/datasets/tbrhznpwg9/1}
# }


def load_weather_data(data_dir='../data/Weather Datasets/Station Wise Data', begin_date='2023-01-01', end_date='2023-08-09'):
    
    import glob
    
    # Convert date strings to datetime
    begin_date = pd.to_datetime(begin_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()
    
    # Get all CSV files in the directory
    csv_files = glob.glob(os.path.join(data_dir, '*.csv'))
    
    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")
    
    print(f"Found {len(csv_files)} CSV files")
    print(f"Date range: {begin_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    
    all_dfs = []
    
    for file_path in csv_files:
        # Extract city name from filename (remove path and .csv extension)
        city_name = os.path.basename(file_path).replace('.csv', '')
        
        # Read the CSV
        df = pd.read_csv(file_path)
        
        # Add City column
        df['City'] = city_name
        
        # Check if year, month, day columns exist
        year_col = None
        month_col = None
        day_col = None
        
        for col in df.columns:
            col_lower = col.lower()
            if 'year' in col_lower:
                year_col = col
            elif 'month' in col_lower:
                month_col = col
            elif 'day' in col_lower:
                day_col = col
        
        # Create Date column from year, month, day
        if year_col and month_col and day_col:
            df['Date'] = pd.to_datetime(
                df[[year_col, month_col, day_col]].astype(int).rename(
                    columns={year_col: 'year', month_col: 'month', day_col: 'day'}
                )
            ).dt.normalize()
            
            # Drop original year, month, day columns
            df = df.drop(columns=[year_col, month_col, day_col])
            
            # Filter by date range
            df = df[(df['Date'] >= begin_date) & (df['Date'] <= end_date)]
        else:
            # Try to find existing date column
            date_col = None
            for col in df.columns:
                if 'date' in col.lower():
                    date_col = col
                    break
            
            if date_col is not None:
                df['Date'] = pd.to_datetime(df[date_col]).dt.normalize()
                if date_col != 'Date':
                    df = df.drop(columns=[date_col])
                # Filter by date range
                df = df[(df['Date'] >= begin_date) & (df['Date'] <= end_date)]
            else:
                print(f"Warning: No date columns found in {city_name}.csv")
        
        all_dfs.append(df)
        # print(f"  ✓ Loaded {city_name}: {len(df)} rows")
    
    # Concatenate all dataframes
    df_weather = pd.concat(all_dfs, ignore_index=True)
    
    # Reorder columns to put Date and City first
    cols = df_weather.columns.tolist()
    priority_cols = ['Date', 'City']
    other_cols = [c for c in cols if c not in priority_cols]
    df_weather = df_weather[priority_cols + other_cols]
    
    # print(f"\n✓ Combined weather data: {len(df_weather)} total rows")
    # print(f"  Cities: {df_weather['City'].nunique()}")
    # print(f"  Date range in data: {df_weather['Date'].min().strftime('%Y-%m-%d')} to {df_weather['Date'].max().strftime('%Y-%m-%d')}")
    # print(f"  Columns: {list(df_weather.columns)}")
    
    return df_weather


# Load weather data for 2023 with specific date range
df_weather = load_weather_data(
    data_dir='../data/Weather Datasets/Station Wise Data', 
    begin_date='2023-01-01', 
    end_date='2023-08-09'
)

df_weather.drop(columns="Unnamed: 0", inplace=True)

df_weather

Found 35 CSV files
Date range: 2023-01-01 to 2023-08-09


,Date,City,Station,Rainfall,Sunshine,Humidity,Temperature
0,2023-01-01,Ambaganctg,Ambaganctg,0.0,8.3,65.0,20.4
1,2023-01-02,Ambaganctg,Ambaganctg,0.0,8.9,73.0,19.9
2,2023-01-03,Ambaganctg,Ambaganctg,0.0,7.6,80.0,19.9
3,2023-01-04,Ambaganctg,Ambaganctg,0.0,9.4,75.0,20.4
4,2023-01-05,Ambaganctg,Ambaganctg,0.0,8.0,78.0,19.6
...,...,...,...,...,...,...,...
7730,2023-08-05,Teknaf,Teknaf,0.0,10.7,84.0,28.8
7731,2023-08-06,Teknaf,Teknaf,2.0,7.2,80.0,29.0
7732,2023-08-07,Teknaf,Teknaf,0.0,7.7,83.0,28.7
7733,2023-08-08,Teknaf,Teknaf,36.0,0.0,92.0,27.5


In [16]:
# Filter weather data for Ambaganctg (case-insensitive)
city_name = "Dhaka" # 'Ambaganctg', "Dhaka"
df_weather[df_weather['City'].astype(str).str.strip().str.lower() == city_name.lower()]

,Date,City,Station,Rainfall,Sunshine,Humidity,Temperature
1989,2023-01-01,Dhaka,Dhaka,0.0,4.0,81.0,17.9
1990,2023-01-02,Dhaka,Dhaka,0.0,5.5,71.0,19.1
1991,2023-01-03,Dhaka,Dhaka,0.0,0.0,87.0,16.2
1992,2023-01-04,Dhaka,Dhaka,0.0,0.0,91.0,14.8
1993,2023-01-05,Dhaka,Dhaka,0.0,1.3,86.0,15.2
...,...,...,...,...,...,...,...
2205,2023-08-05,Dhaka,Dhaka,55.0,5.7,81.0,29.4
2206,2023-08-06,Dhaka,Dhaka,5.0,5.0,91.0,27.8
2207,2023-08-07,Dhaka,Dhaka,18.0,4.2,95.0,27.4
2208,2023-08-08,Dhaka,Dhaka,54.0,7.1,91.0,27.3


<br> <br> <br>

#### 1.3 Create an Event Dataset

In [17]:
df_events = pd.read_csv("../data/BangladeshEvents/event.csv",  parse_dates=["date"])
df_events

,date,category,name,type,scope_or_location,typical_max_temp_C,typical_min_temp_C,typical_rain_mm,weather_basis,impact_on_supply_chain,notes,source_links
0,2023-01-01,school_university,Start of primary & secondary school academic y...,Academic session start,Nationwide,25.4,12.7,7.7,Dhaka Jan climate normals,"School-related demand (uniforms, books, transp...",Academic year in Bangladesh generally runs Jan...,https://dpe.gov.bd/|https://www.unesco.org/en/...
1,2023-01-13,cultural_event,Bishwa Ijtema 2023 – Phase 1 (13–15 Jan),Mass religious congregation (Tablighi Jamaat),Tongi near Dhaka,25.4,12.7,7.7,Dhaka Jan climate normals,Huge temporary population inflow; heavy pressu...,Event spans 3 days; treat as event window in m...,https://www.thedailystar.net/tags/biswa-ijtema
2,2023-01-20,cultural_event,Bishwa Ijtema 2023 – Phase 2 (20–22 Jan),Mass religious congregation,Tongi near Dhaka,25.4,12.7,7.7,Dhaka Jan climate normals,Similar logistical pattern to Phase 1: congest...,Use as second event window close to Dhaka mark...,https://www.thedailystar.net/tags/biswa-ijtema
3,2023-02-14,cultural_event,Pohela Falgun & Valentine's Day,Cultural festival,Urban centres (esp. Dhaka),28.1,15.5,28.9,Dhaka Feb climate normals,"High footfall in parks, campuses, restaurants;...",Good candidate for positive demand shock in ur...,https://www.thedailystar.net/news/bangladesh/n...
4,2023-02-21,national_holiday,Language Martyrs' Day / International Mother L...,Government/National holiday,Nationwide,28.1,15.5,28.9,Dhaka Feb climate normals,Processions and official ceremonies; governmen...,Can model as national_holiday dummy; demand ef...,https://www.officeholidays.com/countries/bangl...
5,2023-03-08,religious_holiday,Shab-e-Barat,Government holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Night prayers and visits to cemeteries; some b...,Mainly timing effect on working hours and urba...,https://calendarific.com/holiday/bangladesh/sh...
6,2023-03-17,national_holiday,Sheikh Mujibur Rahman's Birthday & National Ch...,Government/National holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Government offices closed; ceremonies and gath...,Include as national_holiday dummy.,https://www.officeholidays.com/countries/bangl...
7,2023-03-23,religious_period,Start of Ramadan 1444 (approx.),Religious fasting month,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,"Shift of consumption to evening/night (iftar, ...",Model as month-long behavioural regime change ...,https://www.islamic-relief.org.uk/resources/is...
8,2023-03-26,national_holiday,Independence and National Day,Government/National holiday,Nationwide,32.5,20.4,65.8,Dhaka Mar climate normals,Large rallies and parades; government offices ...,Important national_holiday dummy with possible...,https://www.officeholidays.com/countries/bangl...
9,2023-04-14,cultural_holiday,Bengali New Year (Pohela Boishakh),Government holiday / Cultural new year,Nationwide (strongest in cities),33.7,23.6,156.0,Dhaka Apr climate normals,One of the biggest cultural festivals; very hi...,Strong positive demand shock in retail; possib...,https://www.timeanddate.com/holidays/banglades...


<br> <br> <br>

#### 1.4 Create a Currency Exchange Dataset

In [18]:
df_currency_exchange = pd.read_csv("../data/CurrencyExchange/Bangladeshi_Taka_to_USD.csv",  parse_dates=["Date"])
df_currency_exchange

,Date,BDT_USD
0,2023-01-02 23:58:00,104.7776
1,2023-01-03 23:58:00,101.6990
2,2023-01-04 23:58:00,101.9814
3,2023-01-05 23:58:00,102.0143
4,2023-01-06 23:58:00,102.1778
...,...,...
315,2023-12-25 23:58:00,108.3107
316,2023-12-27 23:58:00,108.0956
317,2023-12-28 23:58:00,108.6817
318,2023-12-29 23:58:00,108.1286


<br> <br> <br>

### 2. Calculate the Coefficient of Variation (CV)

In [19]:
def calculate_product_stats(df, group_col='Product', agg_col='Sales'):
    stats = (
        df.groupby(group_col)
          .agg(
              total_units=(agg_col, 'sum'),
              days_with_activity=(agg_col, lambda x: (x > 0).sum()),
              num_days=(agg_col, 'count'),
              avg_daily=(agg_col, 'mean'),
              std_daily=(agg_col, 'std')
          )
    )
    
    stats['activity_frequency'] = stats['days_with_activity'] / stats['num_days']    
    stats['cv'] = stats['std_daily'] / stats['avg_daily']
    # stats['cv'] = stats['cv'].replace([np.inf, -np.inf], np.nan)
    
    # Reset index to make group_col a regular column
    stats = stats.reset_index()
    
    return stats


# Example usage for all temporal dataframes:

# Sales statistics
sales_stats = calculate_product_stats(df_temporal_sales_order_pivot, group_col='Product', agg_col='Sales')
display(Markdown("**Sales Statistics**"))
display(sales_stats)

**Sales Statistics**

,Product,total_units,days_with_activity,num_days,avg_daily,std_daily,activity_frequency,cv
0,AT5X5K,1.782512e+05,174,221,806.566516,938.839092,0.787330,1.163995
1,ATN01K24P,6.393748e+05,179,221,2893.098569,2701.549761,0.809955,0.933791
2,ATN02K12P,1.370076e+05,177,221,619.943831,615.530467,0.800905,0.992881
3,ATPA1K24P,3.100000e+01,1,221,0.140271,2.085286,0.004525,14.866069
4,ATPPCH5X5K,1.000000e+01,1,221,0.045249,0.672673,0.004525,14.866069
5,ATWWP001K24P,9.223728e+03,159,221,41.736327,52.292598,0.719457,1.252928
6,ATWWP002K12P,3.733000e+03,125,221,16.891401,39.055150,0.565611,2.312132
7,EEA200G24P,5.000000e+00,1,221,0.022624,0.336336,0.004525,14.866069
8,EEA500G12P,1.150000e+02,3,221,0.520362,5.250610,0.013575,10.090302
9,MAC1K25P,8.880000e+02,30,221,4.018100,14.922212,0.135747,3.713749


In [20]:
def classify_speed_segment(stats_df, volume_col='total_units', frequency_col='activity_frequency',  q_high=0.8, q_mid=0.5, freq_threshold=0.5):
    
    df = stats_df.copy()
    
    # Calculate quantiles
    q80 = df[volume_col].quantile(q_high) # e.g. 153888.006 = this value changes based on stats_df
    q50 = df[volume_col].quantile(q_mid)  # e.g. 36836.0001 = this value changes based on stats_df
    print(f"q80 = {q80}")
    print(f"q50 = {q50}")
    print(f"Volume Column is: {volume_col}")
    print(f"if ({volume_col} >= {q80}) and ({frequency_col} >= {freq_threshold}) => `Fast`\nelif {volume_col} >= {q50} => `Medium`\nelse => `Slow`")
    print("\n\n")

    # Classification function
    def classify(row):
        if row[volume_col] >= q80 and row[frequency_col] > freq_threshold: # if total_unit > 153888.006 and "activity_frequency (0.7873)" > 0.4 (0.5=default value) 
            return 'Fast'
        elif row[volume_col] >= q50:
            return 'Medium'
        else:
            return 'Slow'
    
    # Apply classification
    df['speed_segment'] = df.apply(classify, axis=1)
    
    return df



# Classify sales statistics
df_stats_classified = classify_speed_segment(
    sales_stats, 
    volume_col='total_units',
    q_high=0.75,  # Top 25% instead of 20%
    q_mid=0.5,
    freq_threshold=0.4  # Lower frequency threshold
)

df_stats_classified

q80 = 153888.006
q50 = 36836.00000000001
Volume Column is: total_units
if (total_units >= 153888.006) and (activity_frequency >= 0.4) => `Fast`
elif total_units >= 36836.00000000001 => `Medium`
else => `Slow`





,Product,total_units,days_with_activity,num_days,avg_daily,std_daily,activity_frequency,cv,speed_segment
0,AT5X5K,1.782512e+05,174,221,806.566516,938.839092,0.787330,1.163995,Fast
1,ATN01K24P,6.393748e+05,179,221,2893.098569,2701.549761,0.809955,0.933791,Fast
2,ATN02K12P,1.370076e+05,177,221,619.943831,615.530467,0.800905,0.992881,Medium
3,ATPA1K24P,3.100000e+01,1,221,0.140271,2.085286,0.004525,14.866069,Slow
4,ATPPCH5X5K,1.000000e+01,1,221,0.045249,0.672673,0.004525,14.866069,Slow
5,ATWWP001K24P,9.223728e+03,159,221,41.736327,52.292598,0.719457,1.252928,Slow
6,ATWWP002K12P,3.733000e+03,125,221,16.891401,39.055150,0.565611,2.312132,Slow
7,EEA200G24P,5.000000e+00,1,221,0.022624,0.336336,0.004525,14.866069,Slow
8,EEA500G12P,1.150000e+02,3,221,0.520362,5.250610,0.013575,10.090302,Slow
9,MAC1K25P,8.880000e+02,30,221,4.018100,14.922212,0.135747,3.713749,Slow



<br> <br> <br>

#### 2.1 – Identify Product

In [21]:
def get_products_by_speed_segment(df_classified, mover='Fast'):
    """
    Get products filtered by speed segment.
    
    Args:
        df_classified: pandas.DataFrame - classified dataframe with 'speed_segment' column
        mover: str or list - speed segment(s) to filter by
               Can be: 'Fast', 'Medium', 'Slow', 'all'
               Or a list: ['Fast', 'Medium'], ['Fast', 'Slow'], etc.
               
    Returns:
        list: List of unique product names matching the specified segment(s)
    """
    valid_segments = ['Fast', 'Medium', 'Slow']
    
    # Handle 'all' case
    if mover == 'all':
        products = df_classified['Product'].dropna().drop_duplicates().tolist()
        print(f"✓ All products: {len(products)} products")
        return products
    
    # Convert single string to list for uniform processing
    if isinstance(mover, str):
        mover = [mover]
    
    # Validate input
    invalid_segments = [m for m in mover if m not in valid_segments]
    if invalid_segments:
        raise ValueError(f"Invalid segment(s): {invalid_segments}. Valid options: {valid_segments}")
    
    # Filter by speed segment(s)
    mask = df_classified['speed_segment'].isin(mover)
    products = df_classified.loc[mask, 'Product'].dropna().drop_duplicates().tolist()
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"PRODUCTS BY SPEED SEGMENT: {mover}")
    print(f"{'='*70}")
    print(f"Total products: {len(products)}")
    
    # Show breakdown by segment
    for segment in mover:
        segment_count = df_classified[df_classified['speed_segment'] == segment]['Product'].nunique()
        print(f"  - {segment}: {segment_count} products")
    
    # Display product details
    for i, product in enumerate(products, 1):
        stats = df_classified[df_classified['Product'] == product].iloc[0]
        print(f"{i:2d}. {product:15s} | Segment: {stats['speed_segment']:6s} | Volume: {stats['total_units']:>10,.0f} | CV: {stats['cv']:.3f} | Activity: {stats['activity_frequency']:.1%}")
    
    return products


# Example usage:

# Get Fast movers only
# FAST_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Fast')

# Get Medium movers only
# MEDIUM_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Medium')

# Get Slow movers only
# SLOW_MOVERS = get_products_by_speed_segment(df_stats_classified, mover='Slow')

# Get Fast and Medium movers
# FAST_AND_MEDIUM = get_products_by_speed_segment(df_stats_classified, mover=['Fast', 'Medium'])

# Get all products
ALL_PRODUCTS = get_products_by_speed_segment(df_stats_classified, mover='all')

✓ All products: 41 products


<br> <br> <br>

### 3. Prepare Base Dataset - Merging Dataframes

In [22]:
# Filter for fast movers
MOVERS = ALL_PRODUCTS

# Filter for movers
df_sales_mover = df_temporal_sales_order[['Date'] + MOVERS].copy()
df_production_mover = df_temporal_production[['Date'] + [p for p in MOVERS if p in df_temporal_production.columns]].copy()
df_factory_issue_mover = df_temporal_factory_issue[['Date'] + [p for p in MOVERS if p in df_temporal_factory_issue.columns]].copy()
df_delivery_mover = df_temporal_delivery_to_distributor[['Date'] + [p for p in MOVERS if p in df_temporal_delivery_to_distributor.columns]].copy()


# Pivot to long format
df_sales_long = df_sales_mover.melt(id_vars='Date', var_name='Product', value_name='Sales')
df_production_long = df_production_mover.melt(id_vars='Date', var_name='Product', value_name='Production')
df_factory_issue_long = df_factory_issue_mover.melt(id_vars='Date', var_name='Product', value_name='Factory_Issue')
df_delivery_long = df_delivery_mover.melt(id_vars='Date', var_name='Product', value_name='Delivery')

# Merge all temporal data
df_base = df_sales_long.merge(df_production_long, on=['Date', 'Product'], how='left')
df_base = df_base.merge(df_factory_issue_long, on=['Date', 'Product'], how='left')
df_base = df_base.merge(df_delivery_long, on=['Date', 'Product'], how='left')

# Fill NaN with 0 (no activity)
df_base = df_base.fillna(0)

# Sort by Product and Date
df_base = df_base.sort_values(['Product', 'Date']).reset_index(drop=True)

# Add Product, Group and Sub-Group
df_meta_filtered = df_nodes_productgroup_and_subgroup.rename(columns={'Node': 'Product'})
df_base = df_base.merge(df_meta_filtered[['Product', 'Group', 'Sub-Group']], on='Product', how='left')

print(f"Base dataset shape: {df_base.shape}")
print(f"\nColumns: {list(df_base.columns)}")
print(f"\nDate range: {df_base['Date'].min()} to {df_base['Date'].max()}")
print(f"Number of products: {df_base['Product'].nunique()}")
print(f"Total records: {len(df_base):,}")

df_base = df_base.drop_duplicates().reset_index(drop=True)
df_base

Base dataset shape: (9282, 8)

Columns: ['Date', 'Product', 'Sales', 'Production', 'Factory_Issue', 'Delivery', 'Group', 'Sub-Group']

Date range: 2023-01-01 00:00:00 to 2023-08-09 00:00:00
Number of products: 41
Total records: 9,282


,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT
...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS


In [23]:
# Prepare dataframes for merging

# 1. Filter weather data for Dhaka only
df_weather_dhaka = df_weather[df_weather['City'].str.lower() == 'dhaka'].copy()
df_weather_dhaka = df_weather_dhaka.drop(columns=['City'])  # Remove City column since it's always Dhaka

# 2. Prepare events dataframe (rename 'date' to 'Date' for consistency)
df_events_clean = df_events.copy()
df_events_clean = df_events_clean.rename(columns={'date': 'Date'})
df_events_clean['Date'] = pd.to_datetime(df_events_clean['Date']).dt.normalize()

# 3. Prepare currency exchange dataframe
df_currency_clean = df_currency_exchange.copy()
df_currency_clean['Date'] = pd.to_datetime(df_currency_clean['Date']).dt.normalize()

# 4. Prepare holidays dataframe
df_holidays_clean = df_holidays.copy()
df_holidays_clean['Date'] = pd.to_datetime(df_holidays_clean['Date']).dt.normalize()

# 5. Ensure df_base Date is normalized
df_base['Date'] = pd.to_datetime(df_base['Date']).dt.normalize()

# Merge all dataframes
print("Merging dataframes...")

# Merge with holidays
df_merged = df_base.merge(df_holidays_clean, on='Date', how='left')
print(f"  ✓ Merged with holidays: {df_merged.shape}")

# Merge with weather (Dhaka only)
df_merged = df_merged.merge(df_weather_dhaka, on='Date', how='left')
print(f"  ✓ Merged with weather (Dhaka): {df_merged.shape}")

# Merge with events
df_merged = df_merged.merge(df_events_clean, on='Date', how='left')
print(f"  ✓ Merged with events: {df_merged.shape}")

# Merge with currency exchange
df_merged = df_merged.merge(df_currency_clean, on='Date', how='left')
print(f"  ✓ Merged with currency exchange: {df_merged.shape}")

# Fill NaN for categorical columns
if 'Holiday' in df_merged.columns:
    df_merged['Holiday'] = df_merged['Holiday'].fillna('no_holiday')
if 'category' in df_merged.columns:
    df_merged['category'] = df_merged['category'].fillna('no_event')

df_merged

df_merged.drop(columns=["name", "type", "typical_max_temp_C", "typical_min_temp_C", "weather_basis", "impact_on_supply_chain", "notes", "source_links"], inplace=True)

Merging dataframes...
  ✓ Merged with holidays: (9061, 10)
  ✓ Merged with weather (Dhaka): (9061, 15)
  ✓ Merged with events: (9061, 26)
  ✓ Merged with currency exchange: (9061, 27)


In [24]:
df_merged

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798


## 4. Feature Engineering Pipeline

### 4.1 Temporal Features (Lags and Rolling Windows)

In [25]:
def create_lag_features(df, target_col='Sales', lags=[1, 2, 3, 7, 14, 21, 28]):
    """
    Create lag features for each product
    """
    df_feat = df.copy()
    
    for lag in lags:
        df_feat[f'{target_col}_lag_{lag}'] = df_feat.groupby('Product')[target_col].shift(lag)
    
    return df_feat

def create_rolling_features(df, target_col='Sales', windows=[7, 14, 30]):
    """
    Create rolling window statistics
    """
    df_feat = df.copy()
    
    for window in windows:
        # Rolling mean
        df_feat[f'{target_col}_rolling_mean_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        )
        
        # Rolling std
        df_feat[f'{target_col}_rolling_std_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).std())
        )
        
        # Rolling min/max
        df_feat[f'{target_col}_rolling_min_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).min())
        )
        
        df_feat[f'{target_col}_rolling_max_{window}'] = (
            df_feat.groupby('Product')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).max())
        )
    
    return df_feat

# Apply lag features and rolling features for Sales
# print("Creating lag and rolling window features for Sales...")
df_features = create_lag_features(df_merged, target_col='Sales', lags=[1, 2, 3, 7, 14, 21, 28])
# df_features = create_rolling_features(df_features, target_col='Sales', windows=[7, 14, 30])

# Also create features for Production, Factory_Issue, Delivery
# print("Creating lag and rolling window features for Production...")
# df_features = create_lag_features(df_features, target_col='Production', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Production', windows=[1, 7, 14])

# print("Creating lag and rolling window features for Sales...")
# df_features = create_lag_features(df_features, target_col='Factory_Issue', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Factory_Issue', windows=[1, 7, 14])

# print("Creating features for Delivery...")
# df_features = create_lag_features(df_features, target_col='Delivery', lags=[1, 7, 14])
# df_features = create_rolling_features(df_features, target_col='Delivery', windows=[1, 7, 14])

print(f"\n✓ Temporal features created")
print(f"Total features: {df_features.shape[1]}")


✓ Temporal features created
Total features: 26


In [26]:
df_features.head()

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN


In [27]:
def create_supply_chain_features(df):
    """
    Create features capturing supply chain dynamics
    """
    df_feat = df.copy()
    
    # Production-to-sales ratio
    df_feat['production_sales_ratio'] = (df_feat['Production'] / (df_feat['Sales'] + 1e-6))
    
    # Delivery-to-sales ratio
    df_feat['delivery_sales_ratio'] = (df_feat['Delivery'] / (df_feat['Sales'] + 1e-6))
    
    # Factory issue rate (relative to production)
    df_feat['factory_issue_rate'] = (df_feat['Factory_Issue'] / (df_feat['Production'] + 1e-6))
    
    # Cumulative inventory proxy (cumsum of production - sales)
    df_feat['inventory_proxy'] = (
        df_feat.groupby('Product').apply(
            lambda x: (x['Production'] - x['Sales']).cumsum()
        ).reset_index(level=0, drop=True)
    )
    
    # Replace inf with NaN
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
    
    return df_feat

print("Creating supply chain interaction features...")
df_features = create_supply_chain_features(df_features)

print(f"✓ Supply chain features created")
print(f"Total features: {df_features.shape[1]}")

Creating supply chain interaction features...
✓ Supply chain features created
Total features: 30


In [28]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996


In [29]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,BDT_USD,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,104.7776,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,101.6990,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,101.9814,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,102.0143,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,106.5524,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,106.6636,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,107.6734,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,107.7798,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996


<br> <br> <br>

### Merging daily bases datasets from trading economics

In [30]:
def merge_trading_economics_data(df_features, merging_trading_economics_status=False):
    """
    Merge trading economics data with df_features based on Date column.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    merging_trading_economics_status : bool
        If True, merges trading economics dataframes with df_features
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with trading economics data if merging_trading_economics_status is True,
        otherwise returns df_features unchanged
    """
    if not merging_trading_economics_status:
        print("Trading economics merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging trading economics data...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # 1. Merge with currency data (Open, High, Low, Close)
    df_currency_subset = df_trading_economics_currency[['Date', 'Open', 'High', 'Low', 'Close']].copy()
    df_currency_subset['Date'] = pd.to_datetime(df_currency_subset['Date']).dt.normalize()
    df_currency_subset = df_currency_subset.rename(columns={
        'Open': 'currency_open',
        'High': 'currency_high',
        'Low': 'currency_low',
        'Close': 'currency_close'
    })
    df_merged = df_merged.merge(df_currency_subset, on='Date', how='left')
    print(f"  ✓ Merged with currency data: {df_merged.shape}")
    
    # 2. Merge with interest rate data (Value)
    df_interest_subset = df_trading_economics_interest_rate[['DateTime', 'Value']].copy()
    df_interest_subset['DateTime'] = pd.to_datetime(df_interest_subset['DateTime']).dt.normalize()
    df_interest_subset = df_interest_subset.rename(columns={
        'DateTime': 'Date',
        'Value': 'interest_rate_value'
    })
    df_merged = df_merged.merge(df_interest_subset, on='Date', how='left')
    print(f"  ✓ Merged with interest rate data: {df_merged.shape}")
    
    # 3. Merge with stock market data (Open, High, Low, Close)
    df_stock_subset = df_trading_economics_stock_market[['Date', 'Open', 'High', 'Low', 'Close']].copy()
    df_stock_subset['Date'] = pd.to_datetime(df_stock_subset['Date']).dt.normalize()
    df_stock_subset = df_stock_subset.rename(columns={
        'Open': 'stock_market_open',
        'High': 'stock_market_high',
        'Low': 'stock_market_low',
        'Close': 'stock_market_close'
    })
    df_merged = df_merged.merge(df_stock_subset, on='Date', how='left')
    print(f"  ✓ Merged with stock market data: {df_merged.shape}")
    
    print(f"\n✓ Trading economics data merged successfully!")
    print(f"  Added columns: {[col for col in df_merged.columns if col not in df_features.columns]}")
    
    return df_merged


# Example usage:
df_features.drop(columns=["BDT_USD"], inplace=True)
df_features = merge_trading_economics_data(df_features, merging_trading_economics_status=True)

Merging trading economics data...
  ✓ Merged with currency data: (9061, 33)
  ✓ Merged with interest rate data: (9061, 34)
  ✓ Merged with stock market data: (9061, 38)

✓ Trading economics data merged successfully!
  Added columns: ['currency_open', 'currency_high', 'currency_low', 'currency_close', 'interest_rate_value', 'stock_market_open', 'stock_market_high', 'stock_market_low', 'stock_market_close']


In [31]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57


<br> <br> <br>

### Merging inflation rate from trading economics (montly)

we utilized a constant interpolation method for temporal disaggregation, as aligned with the definitions in the Eurostat ESS Guidelines (2018). Since daily inflation data is not available, assigning the official monthly aggregate to each day is the standard baseline approach to align frequencies without introducing artificial variance or unverified assumptions about daily fluctuations.

In [32]:
df_trading_economics_inflation_rate.head()

,Country,Category,DateTime,Value,Frequency,HistoricalDataSymbol,LastUpdate
0,Bangladesh,Inflation Rate,1994-07-31,5.50,Monthly,BangladeshIR,2016-04-05 14:51:00
1,Bangladesh,Inflation Rate,1994-08-31,7.11,Monthly,BangladeshIR,2016-04-05 14:51:00
2,Bangladesh,Inflation Rate,1994-09-30,6.84,Monthly,BangladeshIR,2016-04-05 14:51:00
3,Bangladesh,Inflation Rate,1994-10-31,8.30,Monthly,BangladeshIR,2016-04-05 14:51:00
4,Bangladesh,Inflation Rate,1994-11-30,7.62,Monthly,BangladeshIR,2016-04-05 14:51:00


In [33]:
def merge_inflation_rate_data(df_features, df_inflation_rate, merge_inflation=True):
    """
    Merge monthly inflation rate data with df_features by mapping each month's inflation rate 
    to every single day within that specific month.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    df_inflation_rate : pd.DataFrame
        Inflation rate dataframe with 'DateTime' and 'Value' columns
    merge_inflation : bool
        If True, merges inflation rate data with df_features. If False, returns df_features unchanged (default: True)
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with inflation_rate column if merge_inflation is True,
        otherwise returns df_features unchanged
    """
    if not merge_inflation:
        print("Inflation rate merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging inflation rate data (monthly to daily mapping)...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # Prepare inflation rate data
    df_inflation_subset = df_inflation_rate[['DateTime', 'Value']].copy()
    df_inflation_subset['DateTime'] = pd.to_datetime(df_inflation_subset['DateTime'])
    
    # Extract year and month from inflation data
    df_inflation_subset['Year'] = df_inflation_subset['DateTime'].dt.year
    df_inflation_subset['Month'] = df_inflation_subset['DateTime'].dt.month
    
    # Rename Value column to be more descriptive
    df_inflation_subset = df_inflation_subset.rename(columns={'Value': 'inflation_rate'})
    
    # Keep only Year, Month, and inflation_rate
    df_inflation_subset = df_inflation_subset[['Year', 'Month', 'inflation_rate']]
    
    # Remove duplicates (keep last entry if multiple entries for same month)
    df_inflation_subset = df_inflation_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    print(f"  ✓ Prepared inflation data: {len(df_inflation_subset)} unique months")
    
    # Create temporary Year and Month columns in df_merged
    df_merged['Year'] = df_merged['Date'].dt.year
    df_merged['Month'] = df_merged['Date'].dt.month
    
    # Merge inflation rate data based on Year and Month
    df_merged = df_merged.merge(df_inflation_subset, on=['Year', 'Month'], how='left')
    
    # Drop temporary Year and Month columns
    df_merged = df_merged.drop(columns=['Year', 'Month'])
    
    print(f"  ✓ Merged inflation rate data: {df_merged.shape}")
    print(f"  Missing inflation rate values: {df_merged['inflation_rate'].isna().sum()}")
    
    # Show sample of mapping
    if df_merged['inflation_rate'].notna().any():
        sample_month = df_merged[df_merged['inflation_rate'].notna()]['Date'].dt.to_period('M').iloc[0]
        print(f"\n  Sample: All days in {sample_month} mapped to inflation rate: {df_merged[df_merged['Date'].dt.to_period('M') == sample_month]['inflation_rate'].iloc[0]}")
    
    print(f"\n✓ Inflation rate data merged successfully!")
    
    return df_merged


# Example usage:
# Merge inflation rate data
df_features = merge_inflation_rate_data(
    df_features, 
    df_trading_economics_inflation_rate, 
    merge_inflation=True
)

# Without inflation rate data
# df_features = merge_inflation_rate_data(
#     df_features, 
#     df_trading_economics_inflation_rate, 
#     merge_inflation=False
# )

Merging inflation rate data (monthly to daily mapping)...
  ✓ Prepared inflation data: 378 unique months
  ✓ Merged inflation rate data: (9061, 39)
  Missing inflation rate values: 0

  Sample: All days in 2023-01 mapped to inflation rate: 8.57

✓ Inflation rate data merged successfully!


In [34]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92


<br> <br> <br>

Merge **df_features** with **df_trading_economics_balance_of_trade** and **df_trading_economics_export** and **df_trading_economics_import**


In [35]:
def merge_trade_economics_data(df_features, df_balance_of_trade, df_export, df_import, merge_trade_data=True):
    """
    Merge balance of trade, export, and import data with df_features based on Date column.
    Uses monthly data mapped to each day within that month.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Base dataframe with a 'Date' column
    df_balance_of_trade : pd.DataFrame
        Balance of trade dataframe with 'DateTime' and 'Value' columns
    df_export : pd.DataFrame
        Export dataframe with 'DateTime' and 'Value' columns
    df_import : pd.DataFrame
        Import dataframe with 'DateTime' and 'Value' columns
    merge_trade_data : bool
        If True, merges trade data with df_features. If False, returns df_features unchanged
        
    Returns:
    --------
    pd.DataFrame
        Merged dataframe with trade economics data if merge_trade_data is True,
        otherwise returns df_features unchanged
    """
    if not merge_trade_data:
        print("Trade economics merging is disabled. Returning original df_features.")
        return df_features.copy()
    
    print("Merging trade economics data (monthly to daily mapping)...")
    df_merged = df_features.copy()
    
    # Ensure Date column is datetime and normalized
    df_merged['Date'] = pd.to_datetime(df_merged['Date']).dt.normalize()
    
    # Create temporary Year and Month columns in df_merged
    df_merged['Year'] = df_merged['Date'].dt.year
    df_merged['Month'] = df_merged['Date'].dt.month
    
    # 1. Merge Balance of Trade
    df_balance_subset = df_balance_of_trade[['DateTime', 'Value']].copy()
    df_balance_subset['DateTime'] = pd.to_datetime(df_balance_subset['DateTime'])
    df_balance_subset['Year'] = df_balance_subset['DateTime'].dt.year
    df_balance_subset['Month'] = df_balance_subset['DateTime'].dt.month
    df_balance_subset = df_balance_subset.rename(columns={'Value': 'balance_of_trade'})
    df_balance_subset = df_balance_subset[['Year', 'Month', 'balance_of_trade']]
    df_balance_subset = df_balance_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_balance_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with balance of trade: {df_merged.shape}")
    
    # 2. Merge Exports
    df_export_subset = df_export[['DateTime', 'Value']].copy()
    df_export_subset['DateTime'] = pd.to_datetime(df_export_subset['DateTime'])
    df_export_subset['Year'] = df_export_subset['DateTime'].dt.year
    df_export_subset['Month'] = df_export_subset['DateTime'].dt.month
    df_export_subset = df_export_subset.rename(columns={'Value': 'exports'})
    df_export_subset = df_export_subset[['Year', 'Month', 'exports']]
    df_export_subset = df_export_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_export_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with exports: {df_merged.shape}")
    
    # 3. Merge Imports
    df_import_subset = df_import[['DateTime', 'Value']].copy()
    df_import_subset['DateTime'] = pd.to_datetime(df_import_subset['DateTime'])
    df_import_subset['Year'] = df_import_subset['DateTime'].dt.year
    df_import_subset['Month'] = df_import_subset['DateTime'].dt.month
    df_import_subset = df_import_subset.rename(columns={'Value': 'imports'})
    df_import_subset = df_import_subset[['Year', 'Month', 'imports']]
    df_import_subset = df_import_subset.drop_duplicates(subset=['Year', 'Month'], keep='last')
    
    df_merged = df_merged.merge(df_import_subset, on=['Year', 'Month'], how='left')
    print(f"  ✓ Merged with imports: {df_merged.shape}")
    
    # Drop temporary Year and Month columns
    df_merged = df_merged.drop(columns=['Year', 'Month'])
    
    print(f"\n✓ Trade economics data merged successfully!")
    print(f"  Added columns: balance_of_trade, exports, imports")
    print(f"  Missing balance_of_trade values: {df_merged['balance_of_trade'].isna().sum()}")
    print(f"  Missing exports values: {df_merged['exports'].isna().sum()}")
    print(f"  Missing imports values: {df_merged['imports'].isna().sum()}")
    
    return df_merged


# Merge trade economics data
df_features = merge_trade_economics_data(
    df_features,
    df_trading_economics_balance_of_trade,
    df_trading_economics_export,
    df_trading_economics_import,
    merge_trade_data=True
)

Merging trade economics data (monthly to daily mapping)...
  ✓ Merged with balance of trade: (9061, 42)
  ✓ Merged with exports: (9061, 43)
  ✓ Merged with imports: (9061, 44)

✓ Trade economics data merged successfully!
  Added columns: balance_of_trade, exports, imports
  Missing balance_of_trade values: 0
  Missing exports values: 0
  Missing imports values: 0


In [36]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92,-204.4,386.11,590.46


Based on the code in your notebook, here are the features (columns) selected and merged from each of the specified dataframes into `df_features`. I've listed the original column names from each dataframe and their corresponding renamed feature names in the final dataset:

- **df_trading_economics_currency**:
  - Open → currency_open
  - High → currency_high
  - Low → currency_low
  - Close → currency_close

- **df_trading_economics_stock_market**:
  - Open → stock_market_open
  - High → stock_market_high
  - Low → stock_market_low
  - Close → stock_market_close

- **df_trading_economics_interest_rate**:
  - Value → interest_rate_value

- **df_trading_economics_inflation_rate**:
  - Value → inflation_rate

- **df_trading_economics_balance_of_trade**:
  - Value → balance_of_trade

- **df_trading_economics_export**:
  - Value → exports

- **df_trading_economics_import**:
  - Value → imports

These merges use daily data for currency, stock market, and interest rate (matched on 'Date'), and monthly data for inflation, balance of trade, export, and import (mapped to each day in the month via 'Year' and 'Month'). The 'Date' column is used for alignment, and missing values are handled with left joins.

In [37]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_open,stock_market_high,stock_market_low,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,6177.88,6177.88,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,6185.06,6185.06,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,6202.63,6202.63,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,6193.96,6193.96,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,6299.66,6299.66,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,6315.57,6315.57,6315.57,9.92,-204.4,386.11,590.46


In [38]:
df_features.drop(columns=["stock_market_open", "stock_market_high", "stock_market_low"], inplace=True)

In [39]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,9.92,-204.4,386.11,590.46


In [40]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,currency_open,currency_high,currency_low,currency_close,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,NaN,NaN,NaN,NaN,8.57,-169.1,375.81,544.95
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,103.25,105.95,102.84,105.95,NaN,6177.88,8.57,-169.1,375.81,544.95
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,106.00,106.00,103.26,103.35,NaN,6185.06,8.57,-169.1,375.81,544.95
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,106.00,106.00,103.28,103.30,NaN,6202.63,8.57,-169.1,375.81,544.95
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,106.00,106.00,103.50,103.50,NaN,6193.96,8.57,-169.1,375.81,544.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,NaN,NaN,NaN,NaN,9.92,-204.4,386.11,590.46
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,109.40,109.40,109.33,109.33,NaN,6299.66,9.92,-204.4,386.11,590.46
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,109.40,109.40,109.00,109.40,NaN,6315.57,9.92,-204.4,386.11,590.46


In [41]:
import pandas as pd
import numpy as np

def clean_and_transform_close_only(df_features):
    """
    Isolates currency_close and transforms it for ML readiness.
    """
    # 1. Keep only the essential column
    # We drop others because '0' values are misleading to ML models
    df = df_features.copy()
    
    # 2. Handle missing/zero values in Close
    # Replacing 0 with NaN then forward-filling (common in finance)
    df['currency_close'] = df['currency_close'].replace(0, np.nan)
    df['currency_close'] = df['currency_close'].ffill()
    
    # 3. Log Returns (Scientific requirement for stationarity)
    # Most journals (e.g. Journal of Forecasting) require returns, not prices.
    df['feature_log_return'] = np.log(df['currency_close'] / df['currency_close'].shift(1))
    
    # 4. Momentum Feature (5-day moving average of returns)
    # Research shows 'Return Autocorrelation' is a strong short-term signal.
    df['feature_momentum_5d'] = df['feature_log_return'].rolling(window=5).mean()
    
    # Drop the first few rows that now have NaNs from the shift/rolling operations
    return df

# Usage:
df_final = clean_and_transform_close_only(df_features)

In [42]:
# Usage:

# df_features_copy = clean_and_transform_close_only(df_features)

df_final.drop(columns=["currency_open", "currency_high", "currency_low", "currency_close"], inplace=True)

df_features = df_final.copy()


In [43]:
df_features

,Date,Product,Sales,Production,Factory_Issue,Delivery,Group,Sub-Group,holiday_type,holiday_name,Station,Rainfall,Sunshine,Humidity,Temperature,category,scope_or_location,typical_rain_mm,Sales_lag_1,Sales_lag_2,Sales_lag_3,Sales_lag_7,Sales_lag_14,Sales_lag_21,Sales_lag_28,production_sales_ratio,delivery_sales_ratio,factory_issue_rate,inventory_proxy,interest_rate_value,stock_market_close,inflation_rate,balance_of_trade,exports,imports,feature_log_return,feature_momentum_5d
0,2023-01-01,AT5X5K,2642.0,1500,970.0,707.0,A,AT,NaN,NaN,Dhaka,0.0,4.0,81.0,17.9,school_university,Nationwide,7.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.677517e-01,2.676003e-01,0.646667,-1142.000000,NaN,NaN,8.57,-169.1,375.81,544.95,NaN,NaN
1,2023-01-02,AT5X5K,1070.0,1500,1720.0,2317.0,A,AT,NaN,NaN,Dhaka,0.0,5.5,71.0,19.1,no_event,NaN,NaN,2642.0,NaN,NaN,NaN,NaN,NaN,NaN,1.401869e+00,2.165421e+00,1.146667,-712.000000,NaN,6177.88,8.57,-169.1,375.81,544.95,NaN,NaN
2,2023-01-03,AT5X5K,2355.0,2000,2964.0,2215.6,A,AT,NaN,NaN,Dhaka,0.0,0.0,87.0,16.2,no_event,NaN,NaN,1070.0,2642.0,NaN,NaN,NaN,NaN,NaN,8.492569e-01,9.408068e-01,1.482000,-1067.000000,NaN,6185.06,8.57,-169.1,375.81,544.95,-0.024846,NaN
3,2023-01-04,AT5X5K,909.8,2000,2265.0,2439.0,A,AT,NaN,NaN,Dhaka,0.0,0.0,91.0,14.8,no_event,NaN,NaN,2355.0,1070.0,2642.0,NaN,NaN,NaN,NaN,2.198285e+00,2.680809e+00,1.132500,23.200000,NaN,6202.63,8.57,-169.1,375.81,544.95,-0.000484,NaN
4,2023-01-05,AT5X5K,3504.0,2000,1100.0,735.8,A,AT,NaN,NaN,Dhaka,0.0,1.3,86.0,15.2,no_event,NaN,NaN,909.8,2355.0,1070.0,NaN,NaN,NaN,NaN,5.707763e-01,2.099886e-01,0.550000,-1480.800000,NaN,6193.96,8.57,-169.1,375.81,544.95,0.001934,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9056,2023-08-05,SOS500M24P,109.0,1203,1709.0,1477.0,S,SOS,NaN,NaN,Dhaka,55.0,5.7,81.0,29.4,weather_event,Chattogram division & south-east,316.5,0.0,3669.0,2335.0,0.0,184.0,236.0,513.0,1.103670e+01,1.355046e+01,1.420615,-3956.025996,NaN,NaN,9.92,-204.4,386.11,590.46,0.000000,0.000883
9057,2023-08-06,SOS500M24P,2897.0,1000,1735.0,1179.0,S,SOS,NaN,NaN,Dhaka,5.0,5.0,91.0,27.8,no_event,NaN,NaN,109.0,0.0,3669.0,3810.0,4241.0,4169.0,4971.0,3.451847e-01,4.069727e-01,1.735000,-5853.025996,NaN,NaN,9.92,-204.4,386.11,590.46,0.000000,0.000865
9058,2023-08-07,SOS500M24P,2385.0,2000,1675.0,2506.0,S,SOS,NaN,NaN,Dhaka,18.0,4.2,95.0,27.4,no_event,NaN,NaN,2897.0,109.0,0.0,2454.0,2326.0,2214.0,3291.0,8.385744e-01,1.050734e+00,0.837500,-6238.025996,NaN,6299.66,9.92,-204.4,386.11,590.46,0.003482,-0.000128
9059,2023-08-08,SOS500M24P,0.0,1024,445.0,500.0,S,SOS,NaN,NaN,Dhaka,54.0,7.1,91.0,27.3,no_event,NaN,NaN,2385.0,2897.0,109.0,2528.0,1390.0,1804.0,2124.0,1.024000e+09,5.000000e+08,0.434570,-5214.025996,NaN,6315.57,9.92,-204.4,386.11,590.46,0.000640,0.000824


In [57]:
df_features["typical_rain_mm"].value_counts()





typical_rain_mm
156.0    205
65.8     164
7.7      123
340.0    123
28.9      82
339.0     82
373.0     82
316.5     41
Name: count, dtype: int64

<br> <br> <br>


## Feature Scaling & Encoding

In [ ]:
def scale_and_encode_features(
    df,
    features_for_scaling=None,
    excluded_features=None,
    no_need_for_scaling_features=None,
    categorical_features=None,
    encoding_method='onehot',   # 'onehot', 'label', 'ordinal', or 'none'
    auto_scale_remaining=True,
    remove_excluded=False,
    scale_encoded=False,
    scaler=None,
):
    """
    Scale numeric features and encode categorical features robustly.

    Returns:
        df_processed (pd.DataFrame), processing_info (dict)
    """
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder

    if not isinstance(df, pd.DataFrame):
        raise ValueError("`df` must be a pandas DataFrame")

    excluded_features = excluded_features or []
    no_need_for_scaling_features = no_need_for_scaling_features or []
    categorical_features = categorical_features or []

    # Work on a copy
    df_processed = df.copy()

    # Build sets
    exclude_set = set(excluded_features) | set(no_need_for_scaling_features) | set(categorical_features)

    # Determine numeric features to scale
    if features_for_scaling is None and auto_scale_remaining:
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        features_for_scaling = [c for c in numeric_cols if c not in exclude_set]
    elif features_for_scaling is None:
        features_for_scaling = []

    # Validate and filter scaling features
    valid_scaling_features = [c for c in features_for_scaling if c in df.columns]
    non_numeric = [c for c in valid_scaling_features if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        # Exclude non-numeric items (safety)
        valid_scaling_features = [c for c in valid_scaling_features if c not in non_numeric]

    # Default scaler
    if scaler is None:
        scaler = MinMaxScaler()

    # Scale numeric features (with fallback coercion)
    if valid_scaling_features:
        try:
            df_processed[valid_scaling_features] = scaler.fit_transform(df_processed[valid_scaling_features])
        except Exception:
            # Coerce to numeric and fill NaN with 0 before re-scaling
            for col in valid_scaling_features:
                df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce').fillna(0)
            df_processed[valid_scaling_features] = scaler.fit_transform(df_processed[valid_scaling_features])

    # Encode categorical features
    valid_categorical_features = [c for c in categorical_features if c in df.columns]
    encoders = {}
    encoded_feature_names = []

    if valid_categorical_features and encoding_method is not None and encoding_method.lower() != 'none':
        method = encoding_method.lower()

        if method == 'onehot':
            # Compatibility for sklearn versions with/without sparse_output
            try:
                ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            except TypeError:
                ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')

            # Fit/transform
            cat_block = df_processed[valid_categorical_features].astype(str)
            encoded_arr = ohe.fit_transform(cat_block)

            # If sparse, convert
            if hasattr(encoded_arr, "toarray"):
                encoded_arr = encoded_arr.toarray()

            # Get feature names safely
            try:
                feature_names = list(ohe.get_feature_names_out(valid_categorical_features))
            except Exception:
                # Fallback name generation
                feature_names = []
                for i, col in enumerate(valid_categorical_features):
                    uniques = pd.Index(cat_block[col].unique()).astype(str)
                    for val in uniques:
                        feature_names.append(f"{col}__{val}")

                # Trim/pad to match columns if necessary
                if encoded_arr.shape[1] != len(feature_names):
                    feature_names = [f"ohe_{i}" for i in range(encoded_arr.shape[1])]

            encoded_df = pd.DataFrame(encoded_arr, columns=feature_names, index=df_processed.index)
            df_processed = df_processed.drop(columns=valid_categorical_features)
            df_processed = pd.concat([df_processed, encoded_df], axis=1)

            encoders['onehot'] = ohe
            encoded_feature_names = feature_names

            # Optionally scale encoded features (if requested)
            if scale_encoded and encoded_feature_names:
                enc_scaler = MinMaxScaler()
                df_processed[encoded_feature_names] = enc_scaler.fit_transform(df_processed[encoded_feature_names])
                encoders['scale_encoded'] = enc_scaler

        elif method == 'label':
            for col in valid_categorical_features:
                le = LabelEncoder()
                df_processed[col] = le.fit_transform(df_processed[col].astype(str))
                encoders[f"label_{col}"] = le

        elif method == 'ordinal':
            oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            df_processed[valid_categorical_features] = oe.fit_transform(df_processed[valid_categorical_features].astype(str))
            encoders['ordinal'] = oe

        else:
            raise ValueError("`encoding_method` must be 'onehot', 'label', 'ordinal', or 'none'")

    # Optionally drop excluded columns from returned dataframe
    if remove_excluded:
        to_drop = [c for c in excluded_features if c in df_processed.columns]
        if to_drop:
            df_processed = df_processed.drop(columns=to_drop)

    # Prepare processing info
    processing_info = {
        'scaler': scaler,
        'encoders': encoders,
        'scaled_features': valid_scaling_features,
        'categorical_features': valid_categorical_features,
        'encoded_feature_names': encoded_feature_names,
        'excluded_features': excluded_features,
        'no_scaling_features': no_need_for_scaling_features,
        'encoding_method': encoding_method,
        'auto_scaled': auto_scale_remaining and (features_for_scaling is None)
    }

    return df_processed, processing_info



excluded_features = ["Date", "Station"]
no_need_for_scaling_features = ["feature_log_return", "feature_momentum_5d", "factory_issue_rate", "delivery_sales_ratio", "production_sales_ratio"]
categorical_features = ['Product', "Group", 'Sub-Group', 'holiday_type', "holiday_name", "category", "scope_or_location"]



df_processed, processing_info = scale_and_encode_features(
    df=df_features,
    excluded_features=excluded_features,
    no_need_for_scaling_features=no_need_for_scaling_features,
    categorical_features=categorical_features,
    encoding_method= 'label', # 'onehot',
    remove_excluded=False
)


In [ ]:
df_processed

<br> <br> <br>


## Create Causality Graph

In [ ]:
df_processed.info()

In [ ]:

# =============================================================================
# MAIN FUNCTION: discover_causal_graph
# =============================================================================

def discover_causal_graph(
    df: pd.DataFrame,
    target_col: str,
    exclude_cols: List[str] = [],
    max_lag: int = 1,
    significance_threshold: float = 0.01,
    include_all_nodes: bool = True,
    allow_self_loops: bool = False,
    random_state: int = 42,
    verbose: bool = True
) -> Tuple[nx.DiGraph, Dict[str, Any], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Discover causal relationships in time-series data using VARLiNGAM.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with time-series data
    target_col : str
        Name of the target variable (e.g., 'Sales')
    exclude_cols : List[str]
        Columns to exclude from analysis (e.g., ['Date', 'Station'])
    max_lag : int
        Maximum lag order for VAR model (default: 1)
    significance_threshold : float
        Threshold below which coefficients are considered insignificant (default: 0.01)
    include_all_nodes : bool
        Whether to include all nodes even if they have no edges (default: True)
    allow_self_loops : bool
        Whether to include self-loops in the graph (default: False)
    random_state : int
        Random seed for reproducibility (default: 42)
    verbose : bool
        Print progress and results (default: True)
    
    Returns
    -------
    G : nx.DiGraph
        Causal graph as NetworkX DiGraph
    results : Dict[str, Any]
        Dictionary with model results and metadata
    adj_df : pd.DataFrame
        Combined adjacency matrix as DataFrame
    contemporaneous_df : pd.DataFrame
        DataFrame of contemporaneous effects (Lag 0)
    lagged_df : pd.DataFrame
        DataFrame of lagged effects (Lag 1, 2, ...)
    """
    from lingam import VARLiNGAM
    
    def _print(msg):
        if verbose:
            print(msg)
    
    _print("=" * 60)
    _print("CAUSAL DISCOVERY USING VARLiNGAM")
    _print("=" * 60)
    
    # Get columns to analyze
    analysis_cols = [col for col in df.columns if col not in exclude_cols]
    
    _print(f"\nTarget variable: {target_col}")
    _print(f"Excluded columns: {exclude_cols}")
    _print(f"Max lag: {max_lag}")
    _print(f"Variables for analysis: {len(analysis_cols)}")
    _print(f"Allow self-loops: {allow_self_loops}")
    
    # Prepare data
    df_analysis = df[analysis_cols].copy()
    df_analysis = df_analysis.ffill().bfill()
    for col in df_analysis.columns:
        if df_analysis[col].isna().any():
            df_analysis[col] = df_analysis[col].fillna(df_analysis[col].median())
    
    initial_rows = len(df_analysis)
    df_analysis = df_analysis.dropna()
    _print(f"Rows used: {len(df_analysis)} (dropped {initial_rows - len(df_analysis)})")
    
    X = df_analysis.values
    feature_names = df_analysis.columns.tolist()
    
    _print(f"\nFitting VARLiNGAM model...")
    
    # Fit model
    np.random.seed(random_state)
    model = VARLiNGAM(lags=max_lag, random_state=random_state)
    model.fit(X)
    
    B0 = model.adjacency_matrices_[0]
    B_lags = model.adjacency_matrices_[1:]
    
    _print(f"Model fitted! Shape: {B0.shape}, Lag matrices: {len(B_lags)}")
    
    # Build graph
    G = nx.DiGraph()
    if include_all_nodes:
        for name in feature_names:
            G.add_node(name)
    
    edges_info = []
    contemporaneous_effects = []
    lagged_effects = []
    
    # Process contemporaneous effects (Lag 0)
    for i, tgt in enumerate(feature_names):
        for j, src in enumerate(feature_names):
            # Skip self-loops if not allowed (for contemporaneous, i==j means same variable)
            if not allow_self_loops and i == j:
                continue
            coef = B0[i, j]
            if abs(coef) >= significance_threshold:
                edge_data = {
                    'Source': src, 
                    'Target': tgt, 
                    'Coefficient': coef, 
                    'Abs_Coefficient': abs(coef),
                    'Lag': 0,
                    'Direction': 'Positive' if coef > 0 else 'Negative'
                }
                contemporaneous_effects.append(edge_data)
                edges_info.append({'source': src, 'target': tgt, 'weight': coef, 'lag': 0, 'abs_weight': abs(coef)})
                G.add_edge(src, tgt, weight=coef, lag=0, abs_weight=abs(coef))
    
    # Process lagged effects
    for lag_idx, B_lag in enumerate(B_lags):
        lag = lag_idx + 1
        for i, tgt in enumerate(feature_names):
            for j, src in enumerate(feature_names):
                # For lagged effects, self-loops (i==j) represent autoregressive effects
                # which are often meaningful, but we still respect the allow_self_loops flag
                if not allow_self_loops and i == j:
                    continue
                coef = B_lag[i, j]
                if abs(coef) >= significance_threshold:
                    edge_data = {
                        'Source': src,
                        'Source_Lagged': f"{src}(t-{lag})",
                        'Target': tgt,
                        'Coefficient': coef,
                        'Abs_Coefficient': abs(coef),
                        'Lag': lag,
                        'Direction': 'Positive' if coef > 0 else 'Negative'
                    }
                    lagged_effects.append(edge_data)
                    edges_info.append({'source': src, 'target': tgt, 'weight': coef, 'lag': lag, 'abs_weight': abs(coef)})
                    
                    if G.has_edge(src, tgt):
                        if abs(coef) > G[src][tgt].get('abs_weight', 0):
                            G[src][tgt].update(weight=coef, lag=lag, abs_weight=abs(coef))
                    else:
                        G.add_edge(src, tgt, weight=coef, lag=lag, abs_weight=abs(coef))
    
    # Create DataFrames for contemporaneous and lagged effects
    if contemporaneous_effects:
        contemporaneous_df = pd.DataFrame(contemporaneous_effects)
        contemporaneous_df = contemporaneous_df.sort_values('Abs_Coefficient', ascending=False).reset_index(drop=True)
    else:
        contemporaneous_df = pd.DataFrame(columns=['Source', 'Target', 'Coefficient', 'Abs_Coefficient', 'Lag', 'Direction'])
    
    if lagged_effects:
        lagged_df = pd.DataFrame(lagged_effects)
        lagged_df = lagged_df.sort_values('Abs_Coefficient', ascending=False).reset_index(drop=True)
    else:
        lagged_df = pd.DataFrame(columns=['Source', 'Source_Lagged', 'Target', 'Coefficient', 'Abs_Coefficient', 'Lag', 'Direction'])
    
    # Combined Adjacency DataFrame
    n = len(feature_names)
    combined = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            effects = [B0[i, j]] + [B[i, j] for B in B_lags]
            combined[i, j] = effects[np.argmax(np.abs(effects))]
    
    adj_df = pd.DataFrame(combined, index=feature_names, columns=feature_names)
    
    results = {
        'model': model, 'feature_names': feature_names, 'target_col': target_col,
        'max_lag': max_lag, 'B0': B0, 'B_lags': B_lags, 'edges_info': edges_info,
        'n_nodes': G.number_of_nodes(), 'n_edges': G.number_of_edges(),
        'causal_order': getattr(model, 'causal_order_', None)
    }
    
    _print("\n" + "=" * 60)
    _print("SUMMARY")
    _print("=" * 60)
    _print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    _print(f"Contemporaneous effects (Lag 0): {len(contemporaneous_df)}")
    _print(f"Lagged effects: {len(lagged_df)}")
    
    if target_col in G.nodes():
        _print(f"\nDirect causes of {target_col}: {list(G.predecessors(target_col))}")
        _print(f"Direct effects of {target_col}: {list(G.successors(target_col))}")
    
    return G, results, adj_df, contemporaneous_df, lagged_df


# =============================================================================
# VISUALIZATION: visualize_causal_graph (FIXED - normalized edge widths)
# =============================================================================

def visualize_causal_graph(
    G: nx.DiGraph,
    target_col: str,
    title: str = "Causal Graph (VARLiNGAM)",
    figsize: Tuple[int, int] = (18, 14),
    node_size: int = 2500,
    node_font_size: int = 9,
    min_edge_width: float = 0.5,
    max_edge_width: float = 4.0,
    show_edge_labels: bool = True,
    edge_font_size: int = 7,
    show_self_loops: bool = False,
    legend_fontsize: int = 10,
    layout: str = 'kamada_kawai'
) -> plt.Figure:
    """
    Visualize the causal graph with normalized edge widths.
    
    Parameters
    ----------
    G : nx.DiGraph
        The causal graph
    target_col : str
        Name of the target variable
    title : str
        Title for the plot
    figsize : Tuple[int, int]
        Figure size (width, height)
    node_size : int
        Size of nodes (default: 2500)
    node_font_size : int
        Font size for node labels (default: 9)
    min_edge_width : float
        Minimum edge width after normalization (default: 0.5)
    max_edge_width : float
        Maximum edge width after normalization (default: 4.0)
    show_edge_labels : bool
        Whether to show labels on edges (default: True)
    edge_font_size : int
        Font size for edge labels (default: 7)
    show_self_loops : bool
        Whether to display self-loops (default: False)
    legend_fontsize : int
        Font size for the legend (default: 10)
    layout : str
        Layout algorithm: 'spring', 'circular', 'kamada_kawai', 'shell'
    
    Returns
    -------
    plt.Figure
        The matplotlib figure
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    if len(G.nodes()) == 0:
        ax.text(0.5, 0.5, 'No nodes in graph', ha='center', va='center', fontsize=14)
        ax.axis('off')
        return fig
    
    # Create a copy to optionally filter self-loops
    G_display = G.copy()
    if not show_self_loops:
        self_loops = list(nx.selfloop_edges(G_display))
        G_display.remove_edges_from(self_loops)
    
    # Choose layout
    if layout == 'spring':
        pos = nx.spring_layout(G_display, k=3, iterations=100, seed=42)
    elif layout == 'circular':
        pos = nx.circular_layout(G_display)
    elif layout == 'kamada_kawai':
        try:
            pos = nx.kamada_kawai_layout(G_display)
        except:
            pos = nx.spring_layout(G_display, k=3, iterations=100, seed=42)
    elif layout == 'shell':
        if target_col in G_display.nodes():
            shells = [[target_col]]
            connected = set(G_display.predecessors(target_col)) | set(G_display.successors(target_col)) - {target_col}
            if connected:
                shells.append(list(connected))
            remaining = set(G_display.nodes()) - {target_col} - connected
            if remaining:
                shells.append(list(remaining))
            pos = nx.shell_layout(G_display, nlist=shells)
        else:
            pos = nx.shell_layout(G_display)
    else:
        pos = nx.spring_layout(G_display, k=3, iterations=100, seed=42)
    
    # Node colors
    node_colors = []
    for n in G_display.nodes():
        if n == target_col:
            node_colors.append('#FF6B6B')  # Red for target
        elif G_display.has_edge(n, target_col):
            node_colors.append('#4ECDC4')  # Teal for direct causes
        elif G_display.has_edge(target_col, n):
            node_colors.append('#95E1D3')  # Light green for direct effects
        else:
            node_colors.append('#B8D4E3')  # Light blue for others
    
    # Normalize edge widths to prevent huge edges
    edge_colors = []
    edge_widths = []
    abs_weights = []
    
    for u, v, d in G_display.edges(data=True):
        w = d.get('weight', 0)
        edge_colors.append('#2E86AB' if w > 0 else '#E63946')
        abs_weights.append(abs(w))
    
    # Normalize edge widths
    if abs_weights:
        min_w, max_w = min(abs_weights), max(abs_weights)
        if max_w > min_w:
            edge_widths = [
                min_edge_width + (max_edge_width - min_edge_width) * (w - min_w) / (max_w - min_w)
                for w in abs_weights
            ]
        else:
            edge_widths = [(min_edge_width + max_edge_width) / 2] * len(abs_weights)
    
    # Draw nodes
    nx.draw_networkx_nodes(G_display, pos, node_color=node_colors, node_size=node_size, 
                          alpha=0.9, ax=ax, edgecolors='white', linewidths=2)
    
    # Draw edges
    if G_display.edges():
        nx.draw_networkx_edges(G_display, pos, edge_color=edge_colors, width=edge_widths, alpha=0.7,
                              arrows=True, arrowsize=25, arrowstyle='-|>',
                              connectionstyle='arc3,rad=0.1', ax=ax, 
                              min_source_margin=15, min_target_margin=15)
    
    # Draw node labels
    nx.draw_networkx_labels(G_display, pos, font_size=node_font_size, font_weight='bold', ax=ax)
    
    # Edge labels
    if show_edge_labels and G_display.edges():
        edge_labels = {}
        for u, v, d in G_display.edges(data=True):
            coef = d.get('weight', 0)
            lag = d.get('lag', 0)
            label = f"{coef:.2f}"
            if lag > 0:
                label += f"\n(lag {lag})"
            edge_labels[(u, v)] = label
        
        nx.draw_networkx_edge_labels(G_display, pos, edge_labels=edge_labels, 
                                    font_size=edge_font_size, alpha=0.9, ax=ax,
                                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    # Legend
    legend_elements = [
        plt.scatter([], [], c='#FF6B6B', s=150, label=f'Target ({target_col})', edgecolors='white'),
        plt.scatter([], [], c='#4ECDC4', s=150, label='Direct Causes', edgecolors='white'),
        plt.scatter([], [], c='#95E1D3', s=150, label='Direct Effects', edgecolors='white'),
        plt.scatter([], [], c='#B8D4E3', s=150, label='Other Variables', edgecolors='white'),
        plt.Line2D([0], [0], color='#2E86AB', lw=3, label='Positive Effect'),
        plt.Line2D([0], [0], color='#E63946', lw=3, label='Negative Effect'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=legend_fontsize, framealpha=0.9)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')
    
    plt.tight_layout()
    return fig


# =============================================================================
# VISUALIZATION: visualize_adjacency_heatmap
# =============================================================================

def visualize_adjacency_heatmap(
    adj_df: pd.DataFrame,
    target_col: str,
    title: str = "Causal Adjacency Matrix",
    figsize: Tuple[int, int] = (14, 12)
) -> plt.Figure:
    """Visualize the adjacency matrix as a heatmap."""
    
    fig, ax = plt.subplots(figsize=figsize)
    vmax = max(abs(adj_df.values.min()), abs(adj_df.values.max()))
    if vmax == 0:
        vmax = 1
    
    im = ax.imshow(adj_df.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Causal Effect Coefficient')
    
    ax.set_xticks(range(len(adj_df.columns)))
    ax.set_yticks(range(len(adj_df.index)))
    ax.set_xticklabels(adj_df.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(adj_df.index, fontsize=8)
    
    if target_col in adj_df.index:
        idx = list(adj_df.index).index(target_col)
        for offset in [-0.5, 0.5]:
            ax.axhline(y=idx + offset, color='red', lw=2, ls='--')
            ax.axvline(x=idx + offset, color='red', lw=2, ls='--')
    
    for i in range(len(adj_df)):
        for j in range(len(adj_df)):
            val = adj_df.iloc[i, j]
            if abs(val) >= 0.01:
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6,
                       color='white' if abs(val) > vmax * 0.5 else 'black')
    
    ax.set_xlabel('Cause (Source)', fontsize=11)
    ax.set_ylabel('Effect (Target)', fontsize=11)
    ax.set_title(f"{title}\n(Red lines highlight '{target_col}')", fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig


# =============================================================================
# HELPER: get_causal_effects_on_target
# =============================================================================

def get_causal_effects_on_target(G: nx.DiGraph, results: Dict[str, Any], target_col: str) -> pd.DataFrame:
    """Get a summary DataFrame of all causal effects on the target variable."""
    effects = [{'Cause': e['source'], 'Effect': e['weight'], 'Absolute Effect': e['abs_weight'],
                'Lag': e['lag'], 'Direction': 'Positive' if e['weight'] > 0 else 'Negative'}
               for e in results['edges_info'] if e['target'] == target_col]
    
    if effects:
        return pd.DataFrame(effects).sort_values('Absolute Effect', ascending=False).reset_index(drop=True)
    return pd.DataFrame(columns=['Cause', 'Effect', 'Absolute Effect', 'Lag', 'Direction'])



In [ ]:
# Causal discovery with self-loops
G, results, adj_df, contemporaneous_df, lagged_df = discover_causal_graph(
    df=df_processed,
    target_col='Sales',
    exclude_cols=['Date', 'Station'],
    max_lag=1,
    allow_self_loops=True
)

# Customized visualization
fig = visualize_causal_graph(
    G, 
    target_col='Sales',
    show_edge_labels=False,      # Hide edge labels
    node_size=3000,
    node_font_size=14,
    edge_font_size=8,
    min_edge_width=1.0,
    max_edge_width=5.0,
    show_self_loops=False,
    legend_fontsize=14,  # Increase legend size
    layout = 'shell' # 'spring', 'circular', 'kamada_kawai', 'shell'
)
plt.show()

### Train - Test Split

In [ ]:
df_processed

In [ ]:
# Keep ALL data - no dropping
df_processed.drop(columns=["Station"], inplace=True)
df_clean = df_processed.copy()

# Split based on date only
min_date = df_clean['Date'].min()  # 2023-01-01
max_date = df_clean['Date'].max()  # 2023-08-09
split_date = max_date - timedelta(days=TEST_SIZE_DAYS - 1)

# Split ALL data
train_df = df_clean[df_clean['Date'] < split_date].copy()
test_df = df_clean[df_clean['Date'] >= split_date].copy()

# For target columns: fill NaN with a special value or use masking
target_cols = [col for col in df_clean.columns if col.startswith('target_')]

# # Option 3a: Fill NaN targets with -1 (will be ignored during training)
# for col in target_cols:
#     train_df[col] = train_df[col].fillna(-1)
#     test_df[col] = test_df[col].fillna(-1)

# Option 3b: Create a mask to identify valid targets
# train_df['has_valid_target'] = ~train_df[f'target_{max(FORECAST_HORIZONS)}d_sum'].isna()

print(f"{'='*80}")
print(f"TRAIN-TEST SPLIT (ZERO Data Loss)")
print(f"{'='*80}")
print(f"\nData Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
print(f"Split Date: {split_date.strftime('%Y-%m-%d')}")

print(f"\nTrain Set:")
print(f"  - Date range: {train_df['Date'].min().strftime('%Y-%m-%d')} to {train_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - Shape: {train_df.shape}")
print(f"  - ALL {len(train_df):,} rows kept")

print(f"\nTest Set:")
print(f"  - Date range: {test_df['Date'].min().strftime('%Y-%m-%d')} to {test_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - Shape: {test_df.shape}")
print(f"  - ALL {len(test_df):,} rows kept")

print(f"\nTotal: {len(df_clean):,} samples (100% data usage)")
print(f"Zero rows dropped!")
print(f"\n{'='*80}")

In [ ]:
train_df.shape

In [ ]:
test_df.shape

<br> <br> <br>

### 8. Prepare ML-Ready Datasets

In [ ]:
# Define feature columns (exclude metadata and target columns)
exclude_cols = ["Sales"]  + [col for col in df_clean.columns if col.startswith('target_')] 
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

# Fill remaining NaN with 0 (from lag features at the beginning)
train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

print(f"Feature columns ({len(feature_cols)}):")
print(f"  First 10: {feature_cols[:10]}")
print(f"  Last 10: {feature_cols[-10:]}")

print(f"\nTarget columns ({len([col for col in df_clean.columns if col.startswith('target_')])}):")
print(f"  {[col for col in df_clean.columns if col.startswith('target_')]}")

# Prepare X and y using target_1d_sum
print(f"\n{'='*80}")
print(f"ML-READY DATASETS PREPARED (Using target_1d_sum)")
print(f"{'='*80}")

# Use target_1d_sum for training/testing
# target_col = 'target_1d_sum'
target_col = 'Sales'

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"\nTarget: {target_col}")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")


# Check for -1 values (NaN replacements)
nan_count_train = (y_train == -1).sum()
nan_count_test = (y_test == -1).sum()
print(f"\n  NaN values (marked as -1):")
print(f"    Train: {nan_count_train} ({nan_count_train/len(y_train)*100:.1f}%)")
print(f"    Test: {nan_count_test} ({nan_count_test/len(y_test)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"✓ Feature engineering pipeline complete!")
print(f"  Ready to train with: X_train, y_train, X_test, y_test")
print(f"{'='*80}")

<br> <br> <br>

## Defining ML Metrics

In [ ]:
def calculate_metrics(y_true, y_pred, model_name="Model", y_train=None):
    """Calculate comprehensive evaluation metrics including MASE"""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]
    
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    r2 = r2_score(y_true_clean, y_pred_clean)
    
    # MAPE calculation
    mask_nonzero = y_true_clean != 0
    if mask_nonzero.sum() > 0:
        mape = np.mean(np.abs((y_true_clean[mask_nonzero] - y_pred_clean[mask_nonzero]) / y_true_clean[mask_nonzero])) * 100
    else:
        mape = np.nan
    
    # MASE calculation
    # MASE = MAE / MAE_naive, where naive forecast is y_{t-1}
    if y_train is not None and len(y_train) > 1:
        # Use training data for scaling factor (preferred method)
        y_train_clean = y_train[~np.isnan(y_train)]
        naive_errors = np.abs(np.diff(y_train_clean))
        scale = np.mean(naive_errors)
    elif len(y_true_clean) > 1:
        # Fallback: use test data for scaling factor
        naive_errors = np.abs(np.diff(y_true_clean))
        scale = np.mean(naive_errors)
    else:
        scale = np.nan
    
    if scale > 0 and not np.isnan(scale):
        mase = mae / scale
    else:
        mase = np.nan
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'MASE': mase,
        'R²': r2,
        'N_samples': len(y_true_clean)
    }


def print_metrics_summary(metrics_dict):
    """Print metrics in formatted way"""
    print(f"{metrics_dict['Model']:35s} | RMSE: {metrics_dict['RMSE']:>8,.0f} | MAE: {metrics_dict['MAE']:>8,.0f} | MAPE: {metrics_dict['MAPE']:>6.2f}% | MASE: {metrics_dict['MASE']:>6.3f} | R²: {metrics_dict['R²']:>6.4f}")

print("✓ Helper functions defined")

<br> <br> <br>

## Training Machine Learning Models 

In [ ]:
# Dictionary to store models and results
X_train.drop(columns=["Date"], inplace=True)
X_test.drop(columns=["Date"], inplace=True)

models = {}
predictions = {}
metrics_results = []

print("="*80)
print("TRAINING MULTIPLE MODELS")
print("="*80)

# 1. Random Forest
print("\n1. Training Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

models['RandomForest'] = rf_model
predictions['RandomForest'] = rf_pred

rf_metrics = calculate_metrics(y_test, rf_pred, "RandomForest", y_train=y_train)
metrics_results.append(rf_metrics)
print_metrics_summary(rf_metrics)

# 2. XGBoost
print("\n2. Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05, 
    max_depth=6,
    subsample=0.8, 
    colsample_bytree=0.8, 
    min_child_weight=3,
    reg_alpha=0.1, 
    reg_lambda=1.0, 
    random_state=42,
    n_jobs=-1, 
    tree_method='hist'
)

xgb_model.fit(X_train, y_train, verbose=False)
xgb_pred = xgb_model.predict(X_test)

models['XGBoost'] = xgb_model
predictions['XGBoost'] = xgb_pred

xgb_metrics = calculate_metrics(y_test, xgb_pred, "XGBoost", y_train=y_train)
metrics_results.append(xgb_metrics)
print_metrics_summary(xgb_metrics)

# 3. LightGBM
print("\n3. Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)

models['LightGBM'] = lgb_model
predictions['LightGBM'] = lgb_pred

lgb_metrics = calculate_metrics(y_test, lgb_pred, "LightGBM", y_train=y_train)
metrics_results.append(lgb_metrics)
print_metrics_summary(lgb_metrics)

# 4. CatBoost
print("\n4. Training CatBoost...")
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    subsample=0.8,
    colsample_bylevel=0.8,
    random_state=42,
    verbose=False
)

cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)

models['CatBoost'] = cat_model
predictions['CatBoost'] = cat_pred

cat_metrics = calculate_metrics(y_test, cat_pred, "CatBoost", y_train=y_train)
metrics_results.append(cat_metrics)
print_metrics_summary(cat_metrics)

# Summary comparison
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

# Create summary dataframe
summary_df = pd.DataFrame(metrics_results)
print(summary_df.to_string(index=False))

# Find best model for each metric
print("\n" + "="*80)
print("BEST MODELS BY METRIC")
print("="*80)

# Define metrics and their optimization direction (lower is better or higher is better)
metric_config = {
    'RMSE': 'min',
    'MAE': 'min',
    'MAPE': 'min',
    'MASE': 'min',
    'R²': 'max',
    'SMAPE': 'min'
}

best_models = {}

for metric, direction in metric_config.items():
    if metric in summary_df.columns:
        if direction == 'min':
            best_idx = summary_df[metric].idxmin()
            best_value = summary_df[metric].min()
        else:  # max
            best_idx = summary_df[metric].idxmax()
            best_value = summary_df[metric].max()
        
        best_model_name = summary_df.loc[best_idx, 'Model']
        best_models[metric] = {
            'model': best_model_name,
            'value': best_value
        }
        
        print(f"  {metric:8s}: {best_model_name:15s} ({best_value:.4f})")

# Overall best model (by RMSE as primary metric)
overall_best = best_models['RMSE']['model']
print(f"\n✓ Overall best model (by RMSE): {overall_best}")

print("\n✓ All models trained successfully!")

# Note

## Why MASE needs training data

MASE is defined as:

```
MASE = MAE_model / MAE_naive
```

Where `MAE_naive` is the mean absolute error of a **naive one-step forecast** (predicting y_t = y_{t-1}).

**The standard approach** is to compute `MAE_naive` from the **training data**, not the test data. Here's why:

1. **Fair benchmark** — The naive forecast error on training data represents what a "no-skill" model would achieve on the data your model learned from

2. **Avoids data leakage** — Using test data to compute the scale mixes test information into your metric

3. **Consistent scaling** — All models are compared against the same baseline, regardless of test set characteristics

## Practical example

```python
# During training, you have:
y_train = [100, 105, 103, 110, 108, ...]  # Training targets

# During evaluation:
y_test = [...]   # Actual test values
y_pred = [...]   # Your model's predictions

# Correct usage:
metrics = calculate_metrics(y_test, y_pred, "My Model", y_train=y_train)
```

## When you might skip it

If you don't have access to `y_train` (e.g., evaluating a pre-trained model), the function falls back to using `y_true` (test data) for scaling — this is less ideal but still gives you a usable MASE value.

```python
# Fallback (no training data available):
metrics = calculate_metrics(y_test, y_pred, "My Model")
```

<br> <br> <br>

## Feature Importance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# For explainability
try:
    import lime
    import lime.lime_tabular
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME not available. Install with: pip install lime")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available. Install with: pip install shap")

def explain_model_predictions(X_test, y_test, ml_model, n_samples=5, top_n=10, xai_method='SHAP'):
    
    if xai_method.upper() not in ['LIME', 'SHAP']:
        raise ValueError("xai_method must be 'LIME' or 'SHAP'")
    
    if xai_method.upper() == 'LIME' and not LIME_AVAILABLE:
        raise ImportError("LIME is not installed. Install with: pip install lime")
    
    if xai_method.upper() == 'SHAP' and not SHAP_AVAILABLE:
        raise ImportError("SHAP is not installed. Install with: pip install shap")
    
    # Randomly select n samples
    np.random.seed(42)
    sample_indices = np.random.choice(len(X_test), size=min(n_samples, len(X_test)), replace=False)
    X_samples = X_test.iloc[sample_indices]
    
    print(f"Explaining {len(sample_indices)} random samples using {xai_method.upper()}")
    print(f"Showing top {top_n} features per sample")
    print("="*80)
    
    all_feature_importances = {}
    
    if xai_method.upper() == 'LIME':
        # LIME explanation
        explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=X_test.values,
            feature_names=X_test.columns.tolist(),
            mode='regression',
            random_state=42
        )
        
        for i, idx in enumerate(sample_indices):
            sample = X_samples.iloc[i:i+1]
            
            # Get LIME explanation
            exp = explainer.explain_instance(
                sample.values[0], 
                ml_model.predict, 
                num_features=top_n
            )
            
            # Collect feature importances
            for feature, importance in exp.as_list():
                if feature not in all_feature_importances:
                    all_feature_importances[feature] = []
                all_feature_importances[feature].append(importance)
            
            if i < 3:  # Print details for first 3 samples
                print(f"\nSample {i+1} (Index {idx}) - Top {top_n} features:")
                for feature, importance in exp.as_list():
                    print(f"  {feature}: {importance:.4f}")
    
    elif xai_method.upper() == 'SHAP':
        # SHAP explanation
        if isinstance(ml_model, (RandomForestRegressor, xgb.XGBRegressor, lgb.LGBMRegressor)):
            explainer = shap.TreeExplainer(ml_model)
        elif isinstance(ml_model, CatBoostRegressor):
            explainer = shap.TreeExplainer(ml_model)
        else:
            # For other models, use KernelExplainer (slower)
            background = X_test.sample(min(100, len(X_test)), random_state=42)
            explainer = shap.KernelExplainer(ml_model.predict, background.values)
        
        # Get SHAP values for samples
        shap_values = explainer.shap_values(X_samples.values)
        
        if isinstance(shap_values, list):
            shap_values = shap_values[0]  # For multi-output models
        
        for i, idx in enumerate(sample_indices):
            sample_shap = shap_values[i]
            
            # Get top_n features by absolute SHAP value
            feature_shap = list(zip(X_test.columns, sample_shap))
            top_features = sorted(feature_shap, key=lambda x: abs(x[1]), reverse=True)[:top_n]
            
            # Collect feature importances
            for feature, shap_val in top_features:
                if feature not in all_feature_importances:
                    all_feature_importances[feature] = []
                all_feature_importances[feature].append(shap_val)
            
            if i < 3:  # Print details for first 3 samples
                print(f"\nSample {i+1} (Index {idx}) - Top {top_n} features:")
                for feature, shap_val in top_features:
                    print(f"  {feature}: {shap_val:.4f}")
    
    # Aggregate feature importances (mean across samples)
    aggregated_importances = {}
    for feature, importances in all_feature_importances.items():
        aggregated_importances[feature] = np.mean(importances)
    
    # Create DataFrame with top features
    importance_df = pd.DataFrame(
        list(aggregated_importances.items()),
        columns=['feature_name', 'feature_importance']
    ).sort_values('feature_importance', key=abs, ascending=False)
    
    print(f"\n{'='*80}")
    print(f"AGGREGATED FEATURE IMPORTANCE (Mean across {len(sample_indices)} samples)")
    print(f"{'='*80}")
    print(importance_df.head(20).to_string(index=False))
    
    return importance_df

In [ ]:
# explanations = explain_model_predictions(
#     X_test=X_test,
#     y_test=y_test,
#     ml_model=models['CatBoost'],  # or models['XGBoost'], etc.
#     n_samples=1,
#     top_n=5,
#     xai_method='SHAP'  # or 'LIME'
# )


# # Usage example:

# # Assuming you have trained models and X_test, y_test
# importance_df = explain_model_predictions(
#     X_test=X_test,
#     y_test=y_test,
#     ml_model=models['RandomForest'],  # or models['XGBoost'], etc.
#     n_samples=1,
#     top_n=20,
#     xai_method='SHAP'  # or 'LIME'
# )

# # The returned DataFrame has columns: 'feature_name', 'feature_importance'



importance_df = explain_model_predictions(
    X_test=X_test,
    y_test=y_test,
    ml_model=models['CatBoost'],  # or models['XGBoost'], etc.
    n_samples=1,
    top_n=20,
    xai_method='SHAP'  # or 'LIME'
)

In [ ]:
importance_df = explain_model_predictions(
    X_test=X_test,
    y_test=y_test,
    ml_model=models['CatBoost'],  # or models['XGBoost'], etc.
    n_samples=10,
    top_n=100,
    xai_method='SHAP'  # or 'LIME'
)

In [ ]:
importance_df

In [ ]:
def visualize_feature_importance(importance_df, top_n=20, x_fontsize=10, y_fontsize=10):
    """
    Visualize feature importance with multiple views.
    
    Parameters:
    -----------
    importance_df : pd.DataFrame
        DataFrame with columns 'feature_name' and 'feature_importance'
    top_n : int
        Number of top features to display (default: 20)
    x_fontsize : int
        Font size for x-axis labels (default: 10)
    y_fontsize : int
        Font size for y-axis labels (default: 10)
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette('husl')
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(f'Feature Importance Analysis (Top {top_n} Features)', fontsize=20, fontweight='bold', y=1.00)
    
    # 1. Top N Feature Importances - Horizontal Bar Chart
    ax1 = axes[0, 0]
    top_features = importance_df.head(top_n).sort_values('feature_importance', ascending=True)
    colors = ['green' if x > 0 else 'red' for x in top_features['feature_importance']]
    ax1.barh(range(len(top_features)), top_features['feature_importance'], color=colors, alpha=0.7)
    ax1.set_yticks(range(len(top_features)))
    ax1.set_yticklabels(top_features['feature_name'], fontsize=y_fontsize)
    ax1.set_xlabel('Feature Importance (SHAP Value)', fontsize=12, fontweight='bold')
    ax1.set_title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
    ax1.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax1.grid(axis='x', alpha=0.3)
    ax1.tick_params(axis='x', labelsize=x_fontsize)
    
    # 2. Top 10 Features - Vertical Bar Chart (Absolute Values)
    ax2 = axes[0, 1]
    top_10_display = min(10, top_n)
    top_10_abs = importance_df.head(top_10_display).copy()
    top_10_abs['abs_importance'] = top_10_abs['feature_importance'].abs()
    top_10_abs = top_10_abs.sort_values('abs_importance', ascending=False)
    colors_abs = sns.color_palette('viridis', len(top_10_abs))
    bars = ax2.bar(range(len(top_10_abs)), top_10_abs['abs_importance'], color=colors_abs, alpha=0.8)
    ax2.set_xticks(range(len(top_10_abs)))
    ax2.set_xticklabels(top_10_abs['feature_name'], rotation=45, ha='right', fontsize=x_fontsize)
    ax2.set_ylabel('Absolute Feature Importance', fontsize=12, fontweight='bold')
    ax2.set_title(f'Top {top_10_display} Features by Absolute Importance', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    ax2.tick_params(axis='y', labelsize=y_fontsize)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, top_10_abs['abs_importance'])):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(top_10_abs['abs_importance'])*0.01, 
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # 3. Feature Importance Distribution
    ax3 = axes[1, 0]
    ax3.hist(importance_df['feature_importance'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Line')
    ax3.set_xlabel('Feature Importance Value', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax3.set_title('Distribution of Feature Importances', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(alpha=0.3)
    ax3.tick_params(axis='x', labelsize=x_fontsize)
    ax3.tick_params(axis='y', labelsize=y_fontsize)
    
    # Add statistics text
    mean_imp = importance_df['feature_importance'].mean()
    median_imp = importance_df['feature_importance'].median()
    std_imp = importance_df['feature_importance'].std()
    stats_text = f'Mean: {mean_imp:.4f}\nMedian: {median_imp:.4f}\nStd: {std_imp:.4f}'
    ax3.text(0.02, 0.98, stats_text, transform=ax3.transAxes, 
             fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 4. Cumulative Importance (Top Features)
    ax4 = axes[1, 1]
    cumulative_n = min(30, len(importance_df))
    top_cumulative = importance_df.head(cumulative_n).copy()
    top_cumulative['abs_importance'] = top_cumulative['feature_importance'].abs()
    top_cumulative = top_cumulative.sort_values('abs_importance', ascending=False)
    top_cumulative['cumulative_importance'] = top_cumulative['abs_importance'].cumsum() / top_cumulative['abs_importance'].sum() * 100
    
    ax4.plot(range(1, len(top_cumulative) + 1), top_cumulative['cumulative_importance'], 
             marker='o', linewidth=2, markersize=6, color='darkblue', label='Cumulative %')
    ax4.axhline(y=80, color='red', linestyle='--', linewidth=2, label='80% Threshold')
    ax4.axhline(y=90, color='orange', linestyle='--', linewidth=2, label='90% Threshold')
    ax4.set_xlabel('Number of Top Features', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Cumulative Importance (%)', fontsize=12, fontweight='bold')
    ax4.set_title('Cumulative Feature Importance', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(alpha=0.3)
    ax4.tick_params(axis='x', labelsize=x_fontsize)
    ax4.tick_params(axis='y', labelsize=y_fontsize)
    
    # Find how many features needed for 80% and 90%
    features_80 = (top_cumulative['cumulative_importance'] >= 80).idxmax() + 1 if any(top_cumulative['cumulative_importance'] >= 80) else len(top_cumulative)
    features_90 = (top_cumulative['cumulative_importance'] >= 90).idxmax() + 1 if any(top_cumulative['cumulative_importance'] >= 90) else len(top_cumulative)
    info_text = f'Features for 80%: {features_80}\nFeatures for 90%: {features_90}'
    ax4.text(0.98, 0.02, info_text, transform=ax4.transAxes, 
             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*80)
    print(f"FEATURE IMPORTANCE SUMMARY STATISTICS (Top {top_n} Features)")
    print("="*80)
    print(f"Total features analyzed: {len(importance_df)}")
    print(f"Positive importance features: {(importance_df['feature_importance'] > 0).sum()}")
    print(f"Negative importance features: {(importance_df['feature_importance'] < 0).sum()}")
    print(f"Mean importance: {importance_df['feature_importance'].mean():.6f}")
    print(f"Median importance: {importance_df['feature_importance'].median():.6f}")
    print(f"Std deviation: {importance_df['feature_importance'].std():.6f}")
    print(f"Max importance: {importance_df['feature_importance'].max():.6f} ({importance_df.loc[importance_df['feature_importance'].idxmax(), 'feature_name']})")
    print(f"Min importance: {importance_df['feature_importance'].min():.6f} ({importance_df.loc[importance_df['feature_importance'].idxmin(), 'feature_name']})")
    print("="*80)
    
    # Return top N features as a dataframe
    return importance_df.head(top_n)


# Example usage:
# visualize_feature_importance(importance_df, top_n=20, x_fontsize=10, y_fontsize=10)

# With larger font sizes
visualize_feature_importance(importance_df, top_n=20, x_fontsize=10, y_fontsize=15)

# With smaller font sizes for more features
# visualize_feature_importance(importance_df, top_n=30, x_fontsize=8, y_fontsize=8)

In [ ]:
def visualize_important_features(importance_df, top_n=20, effect_type='both', figsize=(14, 10), fontsize=11, show_values=True, min_threshold=1e-10):
    """
    Visualize important features with a single horizontal bar chart.
    
    Parameters:
    -----------
    importance_df : pd.DataFrame
        DataFrame with columns 'feature_name' and 'feature_importance'
    top_n : int
        Number of top features to display (default: 20)
    effect_type : str
        Type of effects to display: 'positive', 'negative', or 'both' (default: 'both')
    figsize : tuple
        Figure size (width, height) (default: (14, 10))
    fontsize : int
        Font size for labels (default: 11)
    show_values : bool
        Whether to show value labels on bars (default: True)
    min_threshold : float
        Minimum absolute importance threshold to include (default: 1e-10)
        Features with absolute importance below this will be excluded
        
    Returns:
    --------
    pd.DataFrame : DataFrame with the displayed features
    """
    import matplotlib.pyplot as plt
    import pandas as pd
    
    # Filter out features with absolute importance below threshold (effectively 0)
    filtered_df = importance_df[importance_df['feature_importance'].abs() > min_threshold].copy()
    
    # Filter by effect type
    if effect_type == 'positive':
        filtered_df = filtered_df[filtered_df['feature_importance'] > 0.0].copy()
        title_suffix = '(Positive Effects Only)'
        color_scheme = 'green'
    elif effect_type == 'negative':
        filtered_df = filtered_df[filtered_df['feature_importance'] < 0.0].copy()
        title_suffix = '(Negative Effects Only)'
        color_scheme = 'red'
    elif effect_type == 'both':
        title_suffix = '(All Effects)'
        color_scheme = None  # Will use conditional coloring
    else:
        print(f"❌ Error: effect_type must be 'positive', 'negative', or 'both'. Got: {effect_type}")
        return None
    
    # Check if we have data
    if len(filtered_df) == 0:
        print(f"❌ No features found with {effect_type} effects (excluding near-zero effects)")
        return pd.DataFrame()
    
    # Sort by absolute importance and get top N
    filtered_df['abs_importance'] = filtered_df['feature_importance'].abs()
    top_features = filtered_df.nlargest(top_n, 'abs_importance')
    
    # Sort for display (ascending for horizontal bar chart - lowest at bottom)
    top_features = top_features.sort_values('feature_importance', ascending=True)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Determine colors
    if color_scheme:
        colors = [color_scheme] * len(top_features)
    else:
        colors = ['green' if x > 0 else 'red' for x in top_features['feature_importance']]
    
    # Create horizontal bar chart
    bars = ax.barh(range(len(top_features)), top_features['feature_importance'], 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.2)
    
    # Set y-axis labels (feature names)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['feature_name'], fontsize=fontsize)
    
    # Set x-axis label
    ax.set_xlabel('Feature Importance (SHAP Value)', fontsize=fontsize + 2, fontweight='bold')
    
    # Set title
    actual_count = len(top_features)
    title = f'Top {actual_count} Most Important Features {title_suffix}'
    ax.set_title(title, fontsize=fontsize + 4, fontweight='bold', pad=20)
    
    # Add zero line
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.8)
    
    # Add grid
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    # Add value labels on bars if requested
    if show_values:
        for i, (bar, val) in enumerate(zip(bars, top_features['feature_importance'])):
            # Position text at the end of the bar
            if val >= 0:
                x_pos = val
                ha = 'left'
                offset = max(top_features['feature_importance']) * 0.01
            else:
                x_pos = val
                ha = 'right'
                offset = -max(top_features['feature_importance'].abs()) * 0.01
            
            ax.text(x_pos + offset, bar.get_y() + bar.get_height()/2, 
                   f'{val:.4f}', ha=ha, va='center', 
                   fontsize=fontsize - 2, fontweight='bold')
    
    # Add statistics box
    stats_text = (
        f"Total features shown: {len(top_features)}\n"
        f"Positive: {(top_features['feature_importance'] > 0).sum()}\n"
        f"Negative: {(top_features['feature_importance'] < 0).sum()}\n"
        f"Max: {top_features['feature_importance'].max():.4f}\n"
        f"Min: {top_features['feature_importance'].min():.4f}"
    )
    
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
            fontsize=fontsize - 1, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, edgecolor='black'))
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    zero_count = len(importance_df) - len(filtered_df)
    print("\n" + "="*80)
    print(f"FEATURE IMPORTANCE VISUALIZATION SUMMARY")
    print("="*80)
    print(f"Effect type filter: {effect_type}")
    print(f"Features in original dataset: {len(importance_df)}")
    print(f"Features excluded (near-zero): {zero_count}")
    print(f"Features with non-zero effects: {len(filtered_df)}")
    print(f"Features displayed (top {top_n}): {len(top_features)}")
    print(f"  - Positive effects: {(top_features['feature_importance'] > 0).sum()}")
    print(f"  - Negative effects: {(top_features['feature_importance'] < 0).sum()}")
    print("="*80)
    
    # Return the displayed features
    return top_features.sort_values('abs_importance', ascending=False)[['feature_name', 'feature_importance', 'abs_importance']]


# Example usage 1: Show top 20 features with both positive and negative effects
# result = visualize_important_features(importance_df, top_n=50, effect_type='both')

# Example usage 2: Show only top 15 positive effects
# result = visualize_important_features(importance_df, top_n=50, effect_type='positive')

# Example usage 3: Show only top 10 negative effects
# result = visualize_important_features(importance_df, top_n=5, effect_type='negative')

# Example usage 4: Show top 25 features without value labels
# result = visualize_important_features(importance_df, top_n=25, effect_type='both', show_values=True)

# Example usage 5: Custom figure size and font
result = visualize_important_features(importance_df, top_n=20, effect_type='both', figsize=(16, 14), fontsize=9)

<br> <br> <br>

## Generate Sub Graph

In [ ]:

def get_importance_subgraph(
    G: nx.DiGraph,
    importance_df: pd.DataFrame,
    target_col: str,
    feature_col: str = 'feature_name',
    importance_col: str = 'feature_importance',
    top_positive: Optional[int] = None,
    top_negative: Optional[int] = None,
    include_target: bool = True,
    title: str = "Causal Subgraph (Important Features)",
    figsize: Tuple[int, int] = (18, 14),
    node_size: int = 2500,
    node_font_size: int = 9,
    min_edge_width: float = 0.5,
    max_edge_width: float = 4.0,
    show_edge_labels: bool = True,
    edge_font_size: int = 7,
    show_self_loops: bool = False,
    legend_fontsize: int = 10,
    layout: str = 'kamada_kawai'
) -> Tuple[nx.DiGraph, plt.Figure]:
    """
    Extract subgraph based on importance_df and visualize it.
    
    Parameters
    ----------
    G : nx.DiGraph
        The full causal graph from discover_causal_graph()
    importance_df : pd.DataFrame
        DataFrame with feature names and importance scores
    target_col : str
        Name of the target variable (e.g., 'Sales')
    feature_col : str
        Column name for feature names in importance_df (default: 'feature_name')
    importance_col : str
        Column name for importance values in importance_df (default: 'feature_importance')
    top_positive : int or None
        Number of top positive importance features to include (default: None = all positive)
    top_negative : int or None
        Number of top negative importance features to include (default: None = all negative)
    include_target : bool
        Whether to always include the target variable (default: True)
    title : str
        Title for the plot
    figsize : Tuple[int, int]
        Figure size (width, height)
    node_size : int
        Size of nodes (default: 2500)
    node_font_size : int
        Font size for node labels (default: 9)
    min_edge_width : float
        Minimum edge width (default: 0.5)
    max_edge_width : float
        Maximum edge width (default: 4.0)
    show_edge_labels : bool
        Whether to show labels on edges (default: True)
    edge_font_size : int
        Font size for edge labels (default: 7)
    show_self_loops : bool
        Whether to display self-loops (default: False)
    legend_fontsize : int
        Font size for the legend (default: 10)
    layout : str
        Layout algorithm: 'spring', 'circular', 'kamada_kawai', 'shell'
    
    Returns
    -------
    subgraph : nx.DiGraph
        Subgraph containing only features from importance_df
    fig : plt.Figure
        The matplotlib figure
    """
    
    # Filter importance_df based on top_positive and top_negative
    df_filtered = importance_df.copy()
    
    if top_positive is not None or top_negative is not None:
        # Split into positive and negative
        positive_df = df_filtered[df_filtered[importance_col] >= 0].sort_values(
            importance_col, ascending=False
        )
        negative_df = df_filtered[df_filtered[importance_col] < 0].sort_values(
            importance_col, ascending=True  # Most negative first
        )
        
        # Take top N from each
        if top_positive is not None:
            positive_df = positive_df.head(top_positive)
        if top_negative is not None:
            negative_df = negative_df.head(top_negative)
        
        # Combine
        df_filtered = pd.concat([positive_df, negative_df], ignore_index=True)
    
    # Extract features from filtered importance_df
    important_features = set(df_filtered[feature_col].tolist())
    if include_target and target_col:
        important_features.add(target_col)
    
    # Get nodes that exist in both graph and importance_df
    nodes_to_include = important_features & set(G.nodes())
    
    # Create subgraph
    subgraph = G.subgraph(nodes_to_include).copy()
    
    # Remove self-loops if needed
    if not show_self_loops:
        self_loops = list(nx.selfloop_edges(subgraph))
        subgraph.remove_edges_from(self_loops)
    
    print("=" * 60)
    print("SUBGRAPH FROM IMPORTANCE")
    print("=" * 60)
    print(f"Original graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"Features in importance_df: {len(importance_df)}")
    if top_positive is not None:
        print(f"Top positive features: {top_positive}")
    if top_negative is not None:
        print(f"Top negative features: {top_negative}")
    print(f"Features after filtering: {len(df_filtered)}")
    print(f"Subgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges")
    missing = important_features - set(G.nodes())
    if missing:
        print(f"Features not found in graph: {list(missing)}")
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    if subgraph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No matching nodes found', ha='center', va='center', fontsize=14)
        ax.axis('off')
        return subgraph, fig
    
    # Layout
    if layout == 'spring':
        pos = nx.spring_layout(subgraph, k=3, iterations=100, seed=42)
    elif layout == 'circular':
        pos = nx.circular_layout(subgraph)
    elif layout == 'kamada_kawai':
        try:
            pos = nx.kamada_kawai_layout(subgraph)
        except:
            pos = nx.spring_layout(subgraph, k=3, iterations=100, seed=42)
    elif layout == 'shell':
        if target_col in subgraph.nodes():
            shells = [[target_col]]
            connected = set(subgraph.predecessors(target_col)) | set(subgraph.successors(target_col)) - {target_col}
            if connected:
                shells.append(list(connected))
            remaining = set(subgraph.nodes()) - {target_col} - connected
            if remaining:
                shells.append(list(remaining))
            pos = nx.shell_layout(subgraph, nlist=shells)
        else:
            pos = nx.shell_layout(subgraph)
    else:
        pos = nx.spring_layout(subgraph, k=3, iterations=100, seed=42)
    
    # Build importance lookup
    importance_lookup = dict(zip(df_filtered[feature_col], df_filtered[importance_col]))
    
    # Assign colors to nodes based on positive/negative importance
    node_colors = []
    node_color_map = {}
    
    positive_color = '#4ECDC4'  # Teal/Cyan for positive importance
    negative_color = '#FF9500'  # Orange for negative importance
    target_color = '#FF6B6B'    # Red for target
    
    for n in subgraph.nodes():
        if n == target_col:
            node_colors.append(target_color)
            node_color_map[n] = target_color
        else:
            imp = importance_lookup.get(n, 0)
            if imp >= 0:
                node_colors.append(positive_color)
                node_color_map[n] = positive_color
            else:
                node_colors.append(negative_color)
                node_color_map[n] = negative_color
    
    # Edge colors and widths
    edge_colors = []
    abs_weights = []
    for u, v, d in subgraph.edges(data=True):
        w = d.get('weight', 0)
        edge_colors.append('#2E86AB' if w > 0 else '#E63946')
        abs_weights.append(abs(w))
    
    # Normalize edge widths
    edge_widths = []
    if abs_weights:
        min_w, max_w = min(abs_weights), max(abs_weights)
        if max_w > min_w:
            edge_widths = [min_edge_width + (max_edge_width - min_edge_width) * (w - min_w) / (max_w - min_w)
                          for w in abs_weights]
        else:
            edge_widths = [(min_edge_width + max_edge_width) / 2] * len(abs_weights)
    
    # Draw nodes
    nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, node_size=node_size,
                          alpha=0.9, ax=ax, edgecolors='white', linewidths=2)
    
    # Draw edges
    if subgraph.edges():
        nx.draw_networkx_edges(subgraph, pos, edge_color=edge_colors, width=edge_widths, alpha=0.7,
                              arrows=True, arrowsize=25, arrowstyle='-|>',
                              connectionstyle='arc3,rad=0.1', ax=ax,
                              min_source_margin=15, min_target_margin=15)
    
    # Draw labels
    nx.draw_networkx_labels(subgraph, pos, font_size=node_font_size, font_weight='bold', ax=ax)
    
    # Edge labels
    if show_edge_labels and subgraph.edges():
        edge_labels = {}
        for u, v, d in subgraph.edges(data=True):
            coef = d.get('weight', 0)
            lag = d.get('lag', 0)
            label = f"{coef:.2f}"
            if lag > 0:
                label += f"\n(lag {lag})"
            edge_labels[(u, v)] = label
        nx.draw_networkx_edge_labels(subgraph, pos, edge_labels=edge_labels,
                                    font_size=edge_font_size, alpha=0.9, ax=ax,
                                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    # Legend
    legend_elements = [
        plt.scatter([], [], c=target_color, s=150, label=f'Target ({target_col})', edgecolors='white'),
        plt.scatter([], [], c=positive_color, s=150, label='Positive Importance', edgecolors='white'),
        plt.scatter([], [], c=negative_color, s=150, label='Negative Importance', edgecolors='white'),
        plt.Line2D([0], [0], color='#2E86AB', lw=3, label='Positive Effect'),
        plt.Line2D([0], [0], color='#E63946', lw=3, label='Negative Effect'),
    ]
    
    ax.legend(handles=legend_elements, loc='upper left', fontsize=legend_fontsize, framealpha=0.9)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')
    plt.tight_layout()
    
    return subgraph, fig

In [ ]:
# Show top 4 positive and top 4 negative features (8 total + target)
subgraph, fig = get_importance_subgraph(
    G,
    importance_df,
    target_col='Sales',
    top_positive=4,
    top_negative=4
)
plt.show()

# # Show ALL features from importance_df
# subgraph, fig = get_importance_subgraph(
#     G,
#     importance_df,
#     target_col='Sales',
#     top_positive=None,   # None = all positive
#     top_negative=None    # None = all negative
# )
# plt.show()